# Sesión 6 — Cassandra (II), curso 2026-27

![Logotipo de Apache Cassandra](https://raw.githubusercontent.com/dsevilla/bdge/26-27/docs/img/cassandra-logo-notebook.png)

Esta sesión parte de la base de datos de la sesión 5 y la dedica a lo que queda de Cassandra: `GROUP BY`, las escrituras con otra semántica (TTL, escritura condicional y `BATCH`) y, sobre todo, el diseño de tablas dirigido por las consultas, que es lo que practican los ejercicios evaluables.

El cuaderno funciona con un solo nodo (Google Colab o `docker-compose-cassandra.yml`) o con el clúster de dos nodos de `docker-compose-cassandra-2.yml`. Con Docker, arranca antes el Compose y abre el cuaderno en su servicio `notebook` (<http://localhost:8888>, token `bdge`), como explica `cassandra/README.md`: el cuaderno tiene que estar en la red de Compose para llegar a los nodos, y abierto desde fuera la conexión falla tras cinco minutos de reintentos.

## Cómo se usa este boletín

Este boletín es **evaluable** y tiene dos funciones:

1. **Guía de estudio.** Los ejemplos recorren lo que hay que saber de Cassandra para el examen. Las transparencias **«Para el examen»** marcan, sección a sección, qué es lo importante y qué se suele preguntar.
2. **Resultado de entrega.** Los ejercicios del final se realizan en el orden indicado: varios consumen las tablas que dejaron los anteriores, y los parámetros que te corresponden se derivan de tu identificador. Al terminar, una celda lee el resultado final, otra lo envía al registro con el cliente result-only y una tercera permite consultar el ID y la fecha del último envío.

Puedes apoyarte en IA para resolver los ejercicios. El examen y la defensa individual se hacen **sin IA y sin este cuaderno**: lo que cuenta al final es que sepas explicar cada decisión.

### Identificación de la entrega

Escribe tu DNI o NIE en la celda siguiente, tal como figura en tu documento y con o sin la letra final. Su valor **no sale hacia el registro**: lo que se registra es una huella SHA-256 ligada al curso y al boletín. Aun así es un dato pseudonimizado, no anónimo, así que no publiques el cuaderno con tu DNI escrito.

El cuaderno sólo usa los dígitos: si tu identificador empieza por X, Y, Z o U, esa letra se sustituye por un dígito (X→0, Y→1, Z→2, U→3) y la letra final se descarta. Así, `12345678Z` y `12345678` son lo mismo, y `X1234567L` equivale a `01234567`. Tus parámetros salen de esos dígitos. Si el cuaderno rechaza tu identificador, avisa al profesor.

Esta celda sólo deja preparado el cuaderno: no descarga ni envía nada. El identificador se usa más adelante, al llegar a los **ejercicios evaluables**: allí se te vuelve a mostrar para que lo compruebes, se descarga el cliente de entregas, se calcula la huella y se derivan tus parámetros. El envío del resultado es el último paso del boletín.

In [ ]:
# El identificador sólo se usa aquí dentro; lo que se registra es su huella.
CURSO: str = "26-27"
DNI: str = "00000000T"  # Tu DNI o NIE, con o sin letra final
BOLETIN: str = "cassandra-sesion6"

In [ ]:
import sys

RunningInCOLAB: bool = "google.colab" in sys.modules

In [ ]:
if RunningInCOLAB:
    !sudo apt-get update && sudo DEBIAN_FRONTEND=noninteractive apt-get install -y --no-install-recommends default-jdk-headless
    %env JAVA_HOME=/usr/lib/jvm/default-java

## Instalación de Apache Cassandra (solo en Google Colab)

Descargamos el ejecutable comprimido desde su página oficial. Sólo si estamos en Colab. En las ejecuciones con Docker Compose y en las pruebas del CI, Cassandra ya se está ejecutando en otro contenedor o en otra máquina.

In [ ]:
if RunningInCOLAB:
    # Determinar qué versión 5.x.x de Cassandra es la más reciente.
    import re
    from http.client import HTTPResponse
    from urllib.request import Request, urlopen

    from bs4 import BeautifulSoup

    version_re: re.Pattern = re.compile(r"^(5\.[0-9]+(\.[0-9]+)?)")
    req: Request = Request("https://dlcdn.apache.org/cassandra/")
    html_page: HTTPResponse = urlopen(req)

    soup: BeautifulSoup = BeautifulSoup(html_page)
    versions: list[str] = [
        m.group(0) for m in [version_re.match(link.get_text()) for link in soup.find_all("a")] if m
    ]
    if not versions:
        raise RuntimeError(
            "No se encontró ninguna versión 5.x en https://dlcdn.apache.org/cassandra/"
        )
    # Se comparan como números, no como cadenas: "5.0.10" es posterior a "5.0.9".
    bigger5x: str = max(versions, key=lambda version: tuple(int(p) for p in version.split(".")))
    %env CASSANDRA_VER=$bigger5x

Descarga de Cassandra y de su firma digital SHA256 para comprobar que el fichero es correcto. Si es correcto, se descomprime. Esto sólo se realiza en Google Colab. Cada descarga se reintenta con Tenacity (`@retry`): cuatro intentos con esperas de 2, 4 y 8 segundos y aviso antes de cada reintento, y sólo ante errores pasajeros (HTTP 5xx o 429, fallos de red). Si la CDN de Apache sigue fallando, se pide al archivo de Apache.

In [ ]:
if RunningInCOLAB:
    import hashlib
    import logging
    import os
    import shutil
    import tarfile
    from urllib.error import HTTPError, URLError
    from urllib.request import urlopen

    # Esta celda va antes de instalar las dependencias del cuaderno.
    %pip install -q tenacity
    from tenacity import (
        before_sleep_log,
        retry,
        retry_if_exception,
        stop_after_attempt,
        wait_exponential,
    )

    # La CDN de Apache (dlcdn) a veces responde con un error 5xx pasajero
    # («503: first byte timeout»). Cada descarga se reintenta con esperas
    # crecientes y, si la CDN sigue fallando, se pide al archivo de Apache, que
    # guarda todas las versiones. Un 404 u otro error del cliente no se reintenta.
    APACHE_MIRRORS: tuple[str, ...] = (
        "https://dlcdn.apache.org/cassandra",
        "https://archive.apache.org/dist/cassandra",
    )
    DOWNLOAD_ATTEMPTS: int = 4
    DOWNLOAD_RETRY_LOGGER: logging.Logger = logging.getLogger("download_retry")

    def download(path: str, filename: str) -> None:
        """Descarga `path` (relativo a la carpeta de Cassandra) en `filename`."""

        def is_transient(error: BaseException) -> bool:
            """Errores pasajeros: HTTP 5xx o 429 y fallos de red. Un 404 no lo es."""
            if isinstance(error, HTTPError):
                return error.code >= 500 or error.code == 429
            return isinstance(error, (URLError, TimeoutError, ConnectionError))

        # Cuatro intentos con esperas de 2, 4 y 8 s entre ellos.
        @retry(
            retry=retry_if_exception(is_transient),
            stop=stop_after_attempt(DOWNLOAD_ATTEMPTS),
            wait=wait_exponential(multiplier=2),
            before_sleep=before_sleep_log(DOWNLOAD_RETRY_LOGGER, logging.WARNING),
            reraise=True,
        )
        def download_from(url: str, filename: str) -> None:
            """Descarga `url` en `filename`."""
            with urlopen(url, timeout=60) as response, open(filename, "wb") as out:
                shutil.copyfileobj(response, out)

        for mirror in APACHE_MIRRORS:
            try:
                download_from(f"{mirror}/{path}", filename)
                return
            except OSError as error:
                if not is_transient(error):
                    raise
                failure: Exception = error
        raise RuntimeError(f"No se pudo descargar {path} de ningún servidor") from failure

    # Descargar Cassandra y su firma digital SHA256
    cassandra_version: str = os.getenv("CASSANDRA_VER", "5.0.0")
    tarball: str = f"apache-cassandra-{cassandra_version}-bin.tar.gz"
    download(f"{cassandra_version}/{tarball}", tarball)
    download(f"{cassandra_version}/{tarball}.sha256", f"{tarball}.sha256")

    # Comprobar la firma digital SHA256 (file_digest lee el fichero por bloques)
    with open(tarball, "rb") as f:
        actual_hash: str = hashlib.file_digest(f, "sha256").hexdigest()
    with open(f"{tarball}.sha256") as f:
        expected_hash: str = f.read().split()[0]
    if actual_hash != expected_hash:
        raise ValueError("Downloaded Cassandra binary does not match the expected SHA256 hash.")

    # Descomprimir Cassandra
    with tarfile.open(tarball, "r:gz") as tar:
        tar.extractall(filter="data")

Dependiendo de la versión de Java, hay que ejecutar Cassandra con un parámetro adicional.

In [ ]:
if RunningInCOLAB:
    javav = !javap -verbose java.lang.String | grep "major version" | cut -d ' ' -f 5
    cassandra_opts: str = ""
    if int(javav[0]) > 55:
        cassandra_opts = "-Djava.security.manager=allow"
    %env CASSANDRA_OPTS=$cassandra_opts

Usamos la configuración `cassandra_latest.yaml`, la que Cassandra incluye para instalaciones nuevas, con sus funciones más recientes activadas: índices SAI por omisión, entre otras. Cassandra sólo lee `cassandra.yaml`, así que la copiamos encima, igual que hacen los ficheros Compose.

Hay que cambiar además algunas opciones de esa configuración. Son límites de protección pensados para producción. El tamaño a partir del cual un *batch* genera un aviso o se rechaza se relaja, para que las cargas de la hoja no choquen con él:

In [ ]:
if RunningInCOLAB:
    import os
    import re
    import shutil

    conf_dir: str = f"apache-cassandra-{os.environ['CASSANDRA_VER']}/conf"
    config_file: str = f"{conf_dir}/cassandra.yaml"

    # Cassandra sólo lee cassandra.yaml: se copia encima cassandra_latest.yaml, la
    # configuración para instalaciones nuevas, y después se ajusta.
    shutil.copy(f"{conf_dir}/cassandra_latest.yaml", config_file)

    # Read the configuration file
    with open(config_file) as f:
        content: str = f.read()

    # Apply the same replacements as the sed command
    content = re.sub(r"(batch_size_warn_threshold:).*", r"\1 128KiB", content)
    content = re.sub(r"(batch_size_fail_threshold:).*", r"\1 1024KiB", content)

    # Write back the modified content
    with open(config_file, "w") as f:
        f.write(content)

Arrancamos la base de datos (el parámetro `-R` ejecuta el servidor en segundo plano y además permite ejecutarlo como `root`).

In [ ]:
if RunningInCOLAB:
    !(cd apache-cassandra-${CASSANDRA_VER}/ && sudo --preserve-env=MAX_HEAP_SIZE bin/cassandra -R ${CASSANDRA_OPTS})

Con el comando `nodetool status` podemos ver que nuestro clúster de Colab contiene un único *datacenter*, con un único nodo. Hacemos un bucle para esperar hasta que esté disponible, porque la inicialización lleva un tiempo.

In [ ]:
if RunningInCOLAB:
    import os
    import subprocess
    import time

    nodetool: str = f"apache-cassandra-{os.environ['CASSANDRA_VER']}/bin/nodetool"
    deadline: float = time.monotonic() + 600
    while True:
        status: str = subprocess.run(
            [nodetool, "-h", "localhost", "status"], capture_output=True, text=True
        ).stdout
        if any(line.startswith("UN ") for line in status.splitlines()):
            break
        if time.monotonic() > deadline:
            # Si el proceso murió (por ejemplo, sin memoria), el registro dice por qué.
            !tail -n 100 apache-cassandra-${CASSANDRA_VER}/logs/system.log
            raise TimeoutError("Cassandra no arrancó en 10 minutos.")
        time.sleep(5)
    print("Cassandra is up and running.")

Si quisiéramos abrir el shell de comandos de Cassandra para ejecutar instrucciones directamente sobre ella deberíamos ejecutar el siguiente comando.

*De todas formas, dejamos la celda comentada puesto que nuestro interés es en acceder a la BD mediante Python*

In [ ]:
#!apache-cassandra-${CASSANDRA_VER}/bin/cqlsh

## Acceso a Cassandra mediante Python

Instalamos la librería `cassandra-driver` que nos dará acceso a nuestra instancia de Cassandra.

In [ ]:
%pip install cassandra-driver fsspec pyarrow aiohttp tenacity

In [ ]:
# @title Diapositiva: El recorrido de Cassandra (II)
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">El recorrido de Cassandra (II)</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Pasar de consultas a tablas denormalizadas y claves de acceso</li><li style=\'margin:10px 0;list-style-position:outside\'>Ver dónde acaba <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>GROUP BY</code> y dónde se agrega lo demás</li><li style=\'margin:10px 0;list-style-position:outside\'>Conocer TTL, escrituras condicionales y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>BATCH</code></li><li style=\'margin:10px 0;list-style-position:outside\'>Diseñar la clave que sirve a una consulta y reconocer la que no puede servirla</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Idea central: cada optimización cambia las garantías y el coste operativo</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Volver a entrar en Cassandra
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Volver a entrar en Cassandra</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Redescubrir nodos y <em>datacenters</em> con el driver</li><li style=\'margin:10px 0;list-style-position:outside\'>Separar el perfil de carga del de las consultas</li><li style=\'margin:10px 0;list-style-position:outside\'>Observar cómo la topología condiciona consistencia y latencia</li><li style=\'margin:10px 0;list-style-position:outside\'>Reconstruir de una vez la base de datos de la sesión 5</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Antes de optimizar una tabla, hay que saber quién puede responder</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

## Conexión al clúster: puntos de contacto y perfiles de ejecución

Es la misma conexión de la sesión 5, que explica con detalle los puntos de contacto, los *hosts* y las políticas de balanceo. Aquí sólo hace falta recordar tres cosas:

- Los `contact_points` sirven para arrancar: el *driver* descubre el resto de nodos y se conecta a cada uno por la dirección que ese nodo publica.
- Cada consulta elige un **perfil de ejecución** (`ExecutionProfile`), que agrupa balanceo, nivel de consistencia, *timeout* y formato de las filas. No se mezclan con los atributos del modelo antiguo (`session.default_timeout`, `Cluster(load_balancing_policy=...)`), que lanzan `ValueError`.
- Definimos dos perfiles: el de consultas interactivas (el perfil por omisión) y otro para la carga masiva, que admite un nivel de consistencia más débil y un *timeout* mayor.

In [ ]:
# @title Diapositiva: Cuántas réplicas deben responder
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Cuántas réplicas deben responder</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>ONE</code>: basta una réplica; rápido y disponible, pero puede devolver un dato atrasado</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>QUORUM</code>: la mayoría, <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>n // 2 + 1</code>; reconcilia réplicas sin exigir que estén todas</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>ALL</code>: las <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>n</code> réplicas; falla en cuanto cae un nodo</li><li style=\'margin:10px 0;list-style-position:outside\'>Si <strong>R + W &gt; RF</strong>, toda lectura comparte al menos una réplica con la última escritura</li><li style=\'margin:10px 0;list-style-position:outside\'>El perfil por omisión usa <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>LOCAL_QUORUM</code>; la carga masiva, <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>ANY</code></li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>El precio de la consistencia es la disponibilidad</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

### Nivel de consistencia: opciones y cuál usamos

El nivel de consistencia fija cuántas réplicas de una partición deben responder para dar por buena una lectura o una escritura. Con factor de replicación (RF) `n`: `ONE` y `LOCAL_ONE` exigen 1, `QUORUM` y `LOCAL_QUORUM` la mayoría (`n // 2 + 1`), `ALL` las `n`, y `ANY` (sólo escritura) ninguna. Con un único *datacenter*, `LOCAL_` equivale al nivel sin prefijo.

El perfil por omisión usa **`LOCAL_QUORUM`**: con RF = 2 la mayoría es 2, así que R + W = 4 > RF y cada lectura ve lo último que se escribió. Lo necesitan los ejercicios que leen enseguida lo que escribió uno anterior. El precio es la disponibilidad: con RF = 2 falla si cae un nodo. Con un solo nodo (RF = 1) se comporta como `ONE`. La carga masiva usa `ANY`, porque sólo importa la velocidad y nadie lee a mitad de la carga.

In [ ]:
import logging
import os

from cassandra.cluster import (
    EXEC_PROFILE_DEFAULT,
    Cluster,
    ExecutionProfile,
    NoHostAvailable,
    Session,
)
from cassandra.policies import RoundRobinPolicy, TokenAwarePolicy
from cassandra.query import named_tuple_factory
from tenacity import (
    before_sleep_log,
    retry,
    retry_if_exception_type,
    stop_after_delay,
    wait_exponential,
)

from cassandra import ConsistencyLevel

# Puntos de contacto: en Colab, el nodo local; con Docker, el contenido de
# DB_HOSTNAME (que puede llevar varios nombres separados por comas). Sólo se
# usan para arrancar: los demás nodos se descubren leyendo `system.peers`.
contact_points: list[str] = (
    ["localhost"]
    if RunningInCOLAB
    else [name.strip() for name in os.getenv("DB_HOSTNAME", "localhost").split(",") if name.strip()]
)

# Nombre del perfil que usaremos para la carga masiva de datos.
LOAD_PROFILE: str = "load"

# Perfil por omisión: consultas interactivas de la hoja. LOCAL_QUORUM: lecturas y
# escrituras contactan con la mayoría de réplicas y las lecturas reconcilian
# (véase la explicación anterior); con un solo nodo equivale a ONE.
query_profile: ExecutionProfile = ExecutionProfile(
    load_balancing_policy=TokenAwarePolicy(RoundRobinPolicy()),
    consistency_level=ConsistencyLevel.LOCAL_QUORUM,
    request_timeout=40.0,
    row_factory=named_tuple_factory,
)

# Perfil de carga: las escrituras masivas admiten el nivel más débil (ANY, que
# sólo existe para escritura) y necesitan más tiempo por petición. Las lecturas
# posteriores usan el perfil por omisión, que reconcilia las réplicas.
load_profile: ExecutionProfile = ExecutionProfile(
    load_balancing_policy=TokenAwarePolicy(RoundRobinPolicy()),
    consistency_level=ConsistencyLevel.ANY,
    request_timeout=120.0,
    row_factory=named_tuple_factory,
)

cluster: Cluster = Cluster(
    contact_points,
    port=9042,
    execution_profiles={EXEC_PROFILE_DEFAULT: query_profile, LOAD_PROFILE: load_profile},
    # Tras cada CREATE, ALTER o DROP, el driver espera a que todos los nodos tengan
    # la misma versión del esquema antes de devolver el control. Por omisión espera
    # 10 s y, si se agota, sigue sin error: en el clúster de dos nodos no siempre
    # bastaba, y la escritura siguiente llegaba a un nodo que aún no conocía la
    # tabla recién creada (INCOMPATIBLE_SCHEMA).
    max_schema_agreement_wait=30,
)

# Los contenedores pueden tardar en aceptar conexiones, así que `connect()` se
# reintenta con *exponential backoff*: cada intento espera el doble que el
# anterior (1, 2, 4... segundos, con un tope de 30). Si el nodo sólo tardaba un
# momento, se reintenta enseguida; si tarda en arrancar, no se le satura con
# intentos cada pocos segundos. Lo que acota la espera es un plazo total, no un
# número de intentos. Sólo se reintenta `NoHostAvailable`: cualquier otro error
# es un fallo de configuración y debe verse en el acto.
CONNECT_TIMEOUT_S: float = 300.0
CONNECT_RETRY_LOGGER: logging.Logger = logging.getLogger("cassandra_connect_retry")


@retry(
    retry=retry_if_exception_type(NoHostAvailable),
    stop=stop_after_delay(CONNECT_TIMEOUT_S),
    wait=wait_exponential(multiplier=1, max=30),
    before_sleep=before_sleep_log(CONNECT_RETRY_LOGGER, logging.WARNING),
    reraise=True,
)
def connect_to_cluster(cluster: Cluster) -> Session:
    """Abre la sesión del clúster, esperando a que éste acepte conexiones."""
    return cluster.connect()


# Establecer la variable de sesión. Será nuestra conexión a Cassandra.
# Ojo: NO se debe hacer `session.default_timeout = ...` estando en modo de
# perfiles; ese valor es ahora `request_timeout` dentro de cada perfil.
session: Session = connect_to_cluster(cluster)

release_version: str = session.execute("SELECT release_version FROM system.local").one()[0]

print(f"Puntos de contacto: {contact_points}")
print(f"Versión de Cassandra: {release_version}")

Los contenedores pueden tardar en aceptar conexiones, así que `cluster.connect()` va dentro de una función decorada con `@retry` de Tenacity: si el clúster todavía no responde (`NoHostAvailable`), repite el intento con esperas crecientes (1, 2, 4... hasta 30 segundos) durante un máximo de cinco minutos, avisando antes de cada reintento. Cualquier otro error se propaga en el acto.

Una vez conectados, el *driver* ya conoce el clúster completo, aunque le hayamos dado un único punto de contacto. Listamos los nodos con sus metadatos:

In [ ]:
from cassandra.pool import Host

hosts: list[Host] = sorted(cluster.metadata.all_hosts(), key=lambda host: str(host.address))
nhosts: int = len(hosts)

for host in hosts:
    print(
        f"{host.address:<16} dc={host.datacenter} rack={host.rack} "
        f"versión={host.release_version} activo={host.is_up}"
    )
nhosts

### El clúster simulado con contenedores

`docker-compose-cassandra.yml` levanta un nodo y `docker-compose-cassandra-2.yml`, dos (`cassandra1` y `cassandra2`), en la red `cassandra-net`. Cada nodo publica la dirección de su interfaz **dentro de la red de Docker**, así que el cuaderno debe ejecutarse en el servicio `notebook` de ese mismo `docker-compose`; desde el anfitrión sólo funciona `cqlsh`, que habla con un único nodo.

In [ ]:
# @title Diapositiva: La topología ya es parte del modelo
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">La topología ya es parte del modelo</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>El driver descubre hosts y ofrece perfiles con políticas distintas</li><li style=\'margin:10px 0;list-style-position:outside\'>El <em>datacenter</em> local y la consistencia deben ser decisiones conscientes</li><li style=\'margin:10px 0;list-style-position:outside\'>Con Docker, el cuaderno corre dentro de la red de Compose para poder llegar a todos los nodos</li><li style=\'margin:10px 0;list-style-position:outside\'>Con esa base podemos cargar los datos y construir tablas derivadas</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Pasamos de conectividad a datos y patrones de acceso</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

## Datos: la base de datos construida

La sesión 5 construye paso a paso el *keyspace* `stackoverflow` y sus cuatro tablas, y explica cada decisión. Esta sesión parte de la misma base de datos, pero la construye de una vez, sin repetir las explicaciones, para dedicarla a lo nuevo: las claves son las de la sesión 5 y el código es el mismo, con los mismos nombres.

| Tabla | Clave de partición | *Clustering* | Fuente |
|---|---|---|---|
| `Tags` | `Id` | — | `Tags.parquet` |
| `Users` | `Id` | — | `Users.parquet` |
| `Questions` | `Id` | `CreationDate` | `Posts.parquet`, `PostTypeId = 1` |
| `Answers` | `ParentId` | `CreationDate` | `Posts.parquet`, `PostTypeId = 2` |

Los ficheros Parquet no se descargan enteros: se lee su esquema (el *footer*, unos pocos KiB) para generar el CQL, y los datos se recorren por lotes (`RecordBatch`) mientras se insertan. `Posts.parquet` alimenta `Questions` y `Answers` en un solo recorrido.

<details><summary><b>🗂️ Formato de los datos de Stack Overflow</b> <span style="color: #007acc;">▶️ Haz clic para ver el esquema completo</span></summary>

La descripción siguiente resume la [documentación vigente del *data dump* y de SEDE](https://meta.stackexchange.com/questions/2677/database-schema-documentation-for-the-public-data-dump-and-sede/2678). Las prácticas utilizan el *dump* de Stack Overflow en español; SEDE contiene además tablas que no se distribuyen en esos ocho ficheros XML.

> **Notas generales**
>
> - Cada fichero contiene elementos `<row ... />` codificados en UTF-8 y se distribuye dentro de un archivo 7z.
> - Los campos opcionales cuyo valor es nulo no aparecen como atributos en el XML. En los Parquet de la asignatura se representan normalmente como `null`.
> - Las fechas están en UTC y suelen usar el formato `YYYY-MM-DDTHH:mm:ss.fff`.
> - `Id` identifica la fila; los campos indicados como FK hacen referencia al `Id` de otra tabla.

---

:file_folder: <b>Badges.xml</b> — insignias concedidas a usuarios

- `Id`: clave primaria de la concesión.
- `UserId`: usuario que recibe la insignia; FK a `Users.Id`.
- `Name`: nombre de la insignia.
- `Date`: fecha de concesión.
- `Class`: clase de insignia (`1` = oro, `2` = plata, `3` = bronce).
- `TagBased`: `True` si está asociada a una etiqueta; `False` si es una insignia con nombre propio.

---

:file_folder: <b>Comments.xml</b> — comentarios en publicaciones

- `Id`: clave primaria.
- `PostId`: publicación comentada; FK a `Posts.Id`.
- `Score`: puntuación del comentario.
- `Text`: cuerpo del comentario.
- `CreationDate`: fecha de creación.
- `UserDisplayName`: nombre visible del autor cuando no puede referenciarse mediante `UserId`.
- `UserId`: autor; FK opcional a `Users.Id`, ausente si el usuario fue eliminado.
- `ContentLicense`: licencia Creative Commons del contenido.

---

:file_folder: <b>Posts.xml</b> — publicaciones no eliminadas

- `Id`: clave primaria.
- `PostTypeId`: tipo de publicación:
  - `1` pregunta; `2` respuesta; `3` wiki de etiqueta huérfana; `4` extracto de wiki de etiqueta; `5` wiki de etiqueta.
  - `6` candidatura a moderación; `7` marcador wiki; `8` wiki de privilegio; `9` artículo; `10` artículo de ayuda.
  - `12` colección; `13` respuesta a cuestionario de moderación; `14` anuncio; `15` discusión de colectivo; `17` colección de colectivo.
- `AcceptedAnswerId`: respuesta aceptada; FK opcional a `Posts.Id`, sólo para preguntas.
- `ParentId`: pregunta a la que responde; FK a `Posts.Id`, sólo para respuestas.
- `CreationDate`: fecha de creación.
- `Score`: puntuación neta.
- `ViewCount`: número de visitas, opcional.
- `Body`: cuerpo ya renderizado como HTML, no Markdown.
- `OwnerUserId`: autor; FK opcional a `Users.Id`. No aparece si se eliminó el usuario y vale `-1` en las entradas de wiki de etiqueta.
- `OwnerDisplayName`: nombre del autor cuando no hay un usuario referenciable.
- `LastEditorUserId`: último editor; FK opcional a `Users.Id`.
- `LastEditorDisplayName`: nombre del último editor cuando no hay un usuario referenciable.
- `LastEditDate`: fecha de la edición más reciente, opcional.
- `LastActivityDate`: fecha de la actividad más reciente.
- `Title`: título de la pregunta o, en determinados posts de tipo `4`/`5`, nombre de la etiqueta.
- `Tags`: etiquetas de la pregunta o etiqueta asociada a determinados posts de tipo `4`/`5`.
- `AnswerCount`: número de respuestas no eliminadas; sólo aparece en preguntas.
- `CommentCount`: número de comentarios, opcional.
- `FavoriteCount`: número histórico de favoritos, opcional.
- `ClosedDate`: fecha de cierre, sólo si la publicación está cerrada.
- `CommunityOwnedDate`: fecha desde la que la publicación es *community wiki*, si corresponde.
- `ContentLicense`: licencia Creative Commons del contenido.

Desde el *dump* de julio de 2025, `Posts.xml` incluye dos publicaciones artificiales con los identificadores `1000000001` y `1000000010`; no existen en SEDE y pueden excluirse. `DeletionDate` pertenece a `PostsWithDeleted` en SEDE y **no** aparece en el *data dump*.

---

:file_folder: <b>PostHistory.xml</b> — historial de las publicaciones

- `Id`: clave primaria del evento.
- `PostHistoryTypeId`: tipo de evento:
  - `1` título inicial; `2` cuerpo inicial; `3` etiquetas iniciales.
  - `4` edición de título; `5` edición de cuerpo; `6` edición de etiquetas.
  - `7`, `8`, `9`: reversión de título, cuerpo o etiquetas.
  - `10` cierre; `11` reapertura; `12` eliminación; `13` restauración; `14` bloqueo; `15` desbloqueo; `16` pasa a *community wiki*.
  - `17` migración antigua (sustituida por `35`/`36`); `18` fusión; `19` protección; `20` desprotección; `21` desasociación del autor; `22` deshacer fusión.
  - `24` edición sugerida aplicada; `25` publicación tuiteada; `31` discusión de comentarios movida al chat.
  - `33` aviso añadido; `34` aviso retirado; `35` migrada a otro sitio; `36` migrada desde otro sitio; `37` origen de fusión; `38` destino de fusión.
  - `50` subida por el usuario Community; `52` entrada en Hot Network Questions; `53` retirada de HNQ por moderación; `66` creada mediante el asistente de preguntas.
- `PostId`: publicación afectada; FK a `Posts.Id`.
- `RevisionGUID`: agrupa los registros producidos por una misma revisión.
- `CreationDate`: fecha del evento.
- `UserId`: usuario que realizó el cambio; FK opcional a `Users.Id`.
- `UserDisplayName`: nombre visible si el usuario fue eliminado o si se trata del autor de una publicación migrada.
- `Comment`: comentario de edición; para el tipo `10` contiene el motivo de cierre y para `33`/`34`, el identificador del aviso.
- `Text`: valor nuevo en crudo. En ciertos eventos de moderación contiene JSON con los votantes; en duplicados incluye `OriginalQuestionIds`, y para el tipo `17` contiene los detalles de migración.
- `ContentLicense`: licencia Creative Commons del contenido.

---

:file_folder: <b>PostLinks.xml</b> — enlaces entre publicaciones

- `Id`: clave primaria.
- `CreationDate`: fecha de creación del enlace.
- `PostId`: publicación origen; FK a `Posts.Id`.
- `RelatedPostId`: publicación relacionada o destino; FK a `Posts.Id`.
- `LinkTypeId`: tipo de enlace (`1` = enlazada, `3` = duplicada).

---

:file_folder: <b>Users.xml</b> — usuarios

- `Id`: clave primaria.
- `Reputation`: reputación.
- `CreationDate`: fecha de alta.
- `DisplayName`: nombre visible.
- `LastAccessDate`: última carga de una página por el usuario; se actualiza como máximo cada 30 minutos.
- `WebsiteUrl`: sitio web, opcional.
- `Location`: ubicación declarada, opcional.
- `AboutMe`: presentación del perfil, opcional.
- `Views`: visitas al perfil.
- `UpVotes`: votos positivos emitidos.
- `DownVotes`: votos negativos emitidos.
- `ProfileImageUrl`: URL de la imagen de perfil, opcional.
- `EmailHash`: actualmente siempre es nulo y, por tanto, no aparece en el XML.
- `AccountId`: identificador del perfil de red de Stack Exchange; opcional si el usuario oculta esa comunidad en su perfil.

El antiguo campo `Age` ya no forma parte del esquema documentado.

---

:file_folder: <b>Votes.xml</b> — votos y otras acciones sobre publicaciones

- `Id`: clave primaria.
- `PostId`: publicación afectada; FK a `Posts.Id`.
- `VoteTypeId`: tipo de voto o acción:
  - `-1` informar a moderación; `0` deshacer voto de moderación; `1` aceptación por el autor; `2` voto positivo; `3` voto negativo; `4` ofensivo.
  - `5` favorito/marcador (función retirada en octubre de 2022); `6` cierre (desde 2013 se guarda sólo en `PostHistory`); `7` reapertura.
  - `8` inicio de recompensa; `9` fin de recompensa; `10` eliminación; `11` restauración; `12` spam.
  - `15` revisión de moderación; `16` aprobación de edición sugerida; `17`–`21` reacciones/seguimiento; `22`–`28` reacciones de Teams.
  - `29` obsoleta; `30` no obsoleta; `31` voto previo; `32` voto positivo y `33` voto negativo en discusión de colectivo; `35`–`37` evaluación privada de respuesta de IA.
- `UserId`: usuario; sólo está presente para los tipos `5` y `8` (`-1` si se eliminó).
- `CreationDate`: fecha del voto. Por privacidad, la hora se elimina.
- `BountyAmount`: importe de la recompensa; sólo para los tipos `8` y `9`.

No todos los tipos enumerados pueden aparecer en SEDE o en el *dump* público; algunos son exclusivos de Teams o de experimentos concretos.

---

:file_folder: <b>Tags.xml</b> — etiquetas

- `Id`: clave primaria.
- `TagName`: nombre de la etiqueta.
- `Count`: número de preguntas que la usan.
- `ExcerptPostId`: publicación con el extracto de la wiki; FK opcional a `Posts.Id`.
- `WikiPostId`: publicación con el texto de la wiki; FK opcional a `Posts.Id`.
- `IsModeratorOnly`: indica si sólo moderación puede usarla.
- `IsRequired`: indica si su uso es obligatorio.

---

### Diagrama entidad-relación de SEDE

![Diagrama entidad-relación de Stack Exchange Data Explorer](https://raw.githubusercontent.com/dsevilla/bdge/26-27/docs/img/stackexchange-sede-erd.png)

El diagrama corresponde a **SEDE**, cuyo esquema es más rico que los ocho ficheros del *data dump*. Se incluye como visión global de tablas y relaciones; no implica que todas esas tablas estén disponibles en los Parquet de las prácticas. Fuente y autoría: [“The Schema Visualized”](https://meta.stackexchange.com/a/326361).

</details>

In [ ]:
from asyncio import gather, to_thread
from collections.abc import Iterator

import fsspec
import pyarrow as pa
import pyarrow.parquet as pq

PARQUET_BASE_URL: str = (
    "https://github.com/dsevilla/bd2-data/releases/download/es.stackoverflow.data-26-27"
)


def read_remote_parquet_schema(url: str) -> pa.Schema:
    """Lee el esquema Arrow pidiendo por HTTP sólo el *footer* del Parquet."""
    with fsspec.open(url, "rb") as parquet_file:
        return pq.read_schema(parquet_file)


def iter_remote_parquet_batches(url: str, batch_size: int = 20_000) -> Iterator[pa.RecordBatch]:
    """Recorre un Parquet remoto por lotes, descargando sólo lo que va leyendo.

    Es un generador: llamarlo no descarga nada, y el fichero se abre cuando alguien
    pide el primer lote. Así la descarga la hace quien lo recorre (el cargador, en
    su hilo, sin bloquear el cuaderno) y en memoria sólo está lo que se está
    leyendo, no el fichero entero: `Posts` ocupa más de 800 MB como tabla Arrow.
    """
    with fsspec.open(url, "rb") as parquet_file:
        yield from pq.ParquetFile(parquet_file).iter_batches(batch_size=batch_size)


# Sólo los ficheros que alimentan alguna tabla de esta hoja.
PARQUET_NAMES: tuple[str, ...] = ("Posts", "Users", "Tags")

posts_schema: pa.Schema
users_schema: pa.Schema
tags_schema: pa.Schema

posts_schema, users_schema, tags_schema = await gather(
    *(
        to_thread(read_remote_parquet_schema, f"{PARQUET_BASE_URL}/{name}.parquet")
        for name in PARQUET_NAMES
    )
)

In [ ]:
from cassandra.cluster import ResultSet

# El factor de replicación es el número de nodos que ha descubierto el driver,
# con un máximo de 3: un nodo (Colab) tiene RF = 1 y los dos nodos del Compose,
# RF = 2. No se escribe a mano para que el mismo cuaderno valga en los dos casos.
replication_factor: int = min(nhosts, 3)

# SimpleStrategy basta para un solo datacenter y es la más sencilla. En
# producción, y siempre que haya varios datacenters, se usa NetworkTopologyStrategy,
# que fija un factor de replicación por datacenter y reparte las réplicas entre
# racks.
create_keyspace_cql: str = f"""
    CREATE KEYSPACE IF NOT EXISTS stackoverflow
    WITH replication = {{'class': 'SimpleStrategy', 'replication_factor': {replication_factor}}}
"""
print(create_keyspace_cql)

resultado: ResultSet = session.execute(create_keyspace_cql)
print(resultado.one())

In [ ]:
session.set_keyspace("stackoverflow")

`cassandra_create_table_from_pyarrow()` es la función de la sesión 5 que genera el `CREATE TABLE` de cada tabla a partir de su esquema de Arrow: traduce cada tipo a su tipo CQL y compone la clave primaria con `key_form` (primero la clave de partición y después las columnas de *clustering*, en orden).

In [ ]:
from collections import OrderedDict

import pyarrow as pa


def cassandra_create_table_from_pyarrow(
    schema: pa.Schema,
    table_name: str,
    key_form: str | list[str | list[str]] | None = None,
    exclude_fields: list[str] | None = None,
    clustering_order: dict[str, str] | None = None,
    keyspace: str | None = None,
) -> str:
    """
    Generate a CREATE TABLE CQL statement from a PyArrow schema.

    Args:
        schema: pyarrow.Schema instance.
        table_name: target table name (no keyspace).
        key_form: list of lists of column names. First inner list is partition key columns,
                  remaining inner lists are flattened and used as clustering keys in order.
                  Example: [['user_id'], 'bucket', 'created_at']
        exclude_fields: list of field names to skip.
        clustering_order: optional dict mapping clustering column -> "ASC" or "DESC".
        keyspace: optional keyspace name to prefix table.

    Returns:
        cql_create_statement: str
    """

    def _map_arrow_type_to_cassandra(t: pa.DataType) -> str:
        # Handle dictionary (encoded) types
        if pa.types.is_dictionary(t):
            # Get the value type of the dictionary and map that
            value_type = t.value_type
            return _map_arrow_type_to_cassandra(value_type)

        # Handle map types with key-value structure
        if pa.types.is_map(t):
            # Map type: map<key_type, value_type>
            # Recursively map key and value types
            key_type: str = _map_arrow_type_to_cassandra(t.key_type)
            value_type: str = _map_arrow_type_to_cassandra(t.item_type)
            return f"map<{key_type}, {value_type}>"

        # Handle list types
        if pa.types.is_list(t) or pa.types.is_large_list(t):
            # List type: list<element_type>
            element_type: str = _map_arrow_type_to_cassandra(t.value_type)
            return f"list<{element_type}>"

        # Handle struct types
        if pa.types.is_struct(t):
            # Struct type: frozen<map<text, text>> as fallback
            return "frozen<map<text, text>>"

        # primitive types
        if pa.types.is_int8(t):
            return "tinyint"
        if pa.types.is_int16(t):
            return "smallint"
        if pa.types.is_int32(t):
            return "int"
        if pa.types.is_int64(t):
            return "bigint"
        if pa.types.is_uint8(t):
            return "smallint"
        if pa.types.is_uint16(t):
            return "int"
        if pa.types.is_uint32(t):
            return "bigint"
        if pa.types.is_uint64(t):
            return "bigint"
        if pa.types.is_float16(t) or pa.types.is_float32(t):
            return "float"
        if pa.types.is_float64(t):
            return "double"
        if pa.types.is_boolean(t):
            return "boolean"
        if pa.types.is_string(t) or pa.types.is_large_string(t):
            return "text"
        if pa.types.is_binary(t) or pa.types.is_large_binary(t) or pa.types.is_fixed_size_binary(t):
            return "blob"
        if pa.types.is_timestamp(t):
            return "timestamp"
        if pa.types.is_date32(t) or pa.types.is_date64(t):
            return "date"
        if pa.types.is_time32(t) or pa.types.is_time64(t):
            return "time"
        if pa.types.is_decimal(t):
            return "decimal"
        # fallback
        return "text"

    exclude: set[str] = set(exclude_fields or [])
    columns: OrderedDict[str, str] = OrderedDict()
    for fld in schema:
        if fld.name in exclude:
            continue
        cass_t: str = _map_arrow_type_to_cassandra(fld.type)
        columns[fld.name] = cass_t

    if not columns:
        raise ValueError("No columns left after applying exclude_fields.")

    # Build PRIMARY KEY
    if key_form:
        if not isinstance(key_form, (str, list)) or not all(
            isinstance(k, (str, list)) for k in key_form
        ):
            raise ValueError("Invalid key_form format.")

        match key_form:
            case str():  # "key"
                partition_cols: list[str] = [key_form]
                clustering_cols: list[str] = []
            case list():  # ["<key>", "<clustering>"] or [["<key1>", "<key2>", ...], "clustering"]
                match key_form[0]:
                    case list():
                        partition_cols = key_form[0]
                    case str():
                        partition_cols = [key_form[0]]
                clustering_cols: list[str] = key_form[1:]  # type: ignore
    else:
        field_names: list[str] = [f.name for f in schema if f.name not in exclude]
        partition_cols = [field_names[0]]
        clustering_cols = field_names[1:]

    for col in partition_cols + clustering_cols:
        if col not in columns:
            raise ValueError(f"Primary key column '{col}' not found in schema (or was excluded).")

    cols_cql_lines: list[str] = [f"  {name} {typ}" for name, typ in columns.items()]
    cols_cql_str: str = ",\n".join(cols_cql_lines)

    if len(partition_cols) == 1:
        pk_left: str = partition_cols[0]
    else:
        pk_left = "(" + ", ".join(partition_cols) + ")"
    pk_rest: str = ", ".join(clustering_cols) if clustering_cols else ""
    pk_clause: str = f"PRIMARY KEY ({pk_left}" + ((", " + pk_rest) if pk_rest else "") + ")"

    clustering_clause = ""
    if clustering_cols and clustering_order:
        orders: list[str] = []
        for c in clustering_cols:
            ordv: str | None = clustering_order.get(c)
            if ordv:
                ordv_upper = ordv.upper()
                if ordv_upper not in ("ASC", "DESC"):
                    raise ValueError("clustering_order values must be 'ASC' or 'DESC'")
                orders.append(f"{c} {ordv_upper}")
        if orders:
            clustering_clause: str = " \nAND CLUSTERING ORDER BY (" + ", ".join(orders) + ")"

    ks_prefix: str = f"{keyspace}." if keyspace else ""
    create: str = (
        f"CREATE TABLE IF NOT EXISTS {ks_prefix}{table_name} (\n{cols_cql_str},\n  {pk_clause}\n)"
        + " WITH compression = {'class': 'LZ4Compressor'} "
        + f"{clustering_clause};"
    )
    return create


def cassandra_insert_template(
    schema: pa.Schema,
    table_name: str,
    exclude_fields: set[str] | None = None,
    keyspace: str | None = None,
) -> tuple[str, list[str]]:
    """
    Construye una plantilla CQL de INSERT para Cassandra (placeholders posicionales '?') a partir
    de un esquema PyArrow.

    Devuelve (insert_cql, columns), donde:
      - insert_cql es, por ejemplo, "INSERT INTO ks.table (c1,c2) VALUES (?,?);"
      - columns son las columnas del INSERT, en el orden definido por el esquema: los
        valores deben enviarse en ese orden.

    Notas:
      - Los campos indicados en exclude_fields se omiten.
      - No realiza mapeo de tipos ni prepara la sentencia contra un clúster.
    """
    exclude: set[str] = exclude_fields or set()
    cols: list[str] = [field.name for field in schema if field.name not in exclude]

    if not cols:
        raise ValueError("No columns remain after applying exclude_fields.")

    ks_prefix: str = f"{keyspace}." if keyspace else ""
    col_list: str = ", ".join(cols)
    placeholders: str = ", ".join("?" for _ in cols)
    cql: str = f"INSERT INTO {ks_prefix}{table_name} ({col_list}) VALUES ({placeholders});"
    return cql, cols

`populate_table()` es la función de la sesión 5 que escribe en una tabla, de forma concurrente, una fila por cada fila de origen:

```python
filas_escritas = populate_table(session, statement, rows, execution_profile=..., to_values=None, concurrency=100)
```

- `statement`: el `INSERT` o `UPDATE` preparado; cada fila aporta los valores de sus `?`, en el mismo orden.
- `rows`: cualquier iterable (una lista de tuplas, un generador o el `ResultSet` de un `SELECT`), que se recorre una sola vez, sin guardarlo.
- `execution_profile` (obligatorio): `LOAD_PROFILE` para una carga masiva, o `EXEC_PROFILE_DEFAULT` (`LOCAL_QUORUM`) cuando lo escrito se va a leer enseguida, porque `ANY` no garantiza esa lectura.
- `to_values` (opcional): convierte cada fila de origen en sus valores, o devuelve `None` para saltarla.
- `concurrency` (opcional): cuántas escrituras puede haber en vuelo a la vez.

Devuelve el número de filas escritas. Por dentro, recorre `rows` en el hilo principal y envía con `execute_async()`, con `concurrency` escrituras en vuelo como mucho. No usa `execute_concurrent_with_args()`, que pide cada fila desde el hilo de red del *driver*: con un generador que tenga que traer una página, ese hilo se quedaría colgado.

In [ ]:
from collections import deque
from collections.abc import Callable, Iterable, Sequence
from typing import Any

from cassandra.cluster import ResponseFuture, Session
from cassandra.query import PreparedStatement


def populate_table(
    session: Session,
    statement: PreparedStatement,
    rows: Iterable[Any],
    *,
    execution_profile: str,
    to_values: Callable[[Any], Sequence[Any] | None] | None = None,
    concurrency: int = 100,
) -> int:
    """Escribe con `statement`, de forma concurrente, una fila por cada fila de `rows`.

    `rows` es cualquier iterable (una lista, un generador, el `ResultSet` de un
    `SELECT`), que se recorre una sola vez, sin guardarlo. `to_values` convierte
    cada fila de origen en los valores de los `?` de `statement`, o devuelve
    `None` para saltarla; sin ella, se envía tal cual.
    Devuelve el número de filas escritas.
    """
    in_flight: deque[ResponseFuture] = deque()
    written: int = 0
    for row in rows:
        row_values: Sequence[Any] | None = row if to_values is None else to_values(row)
        if row_values is None:
            continue
        # Con el cupo lleno, se espera a la escritura más antigua antes de enviar
        # otra. result() lanza la excepción de la escritura si ha fallado.
        if len(in_flight) == concurrency:
            in_flight.popleft().result()
        in_flight.append(
            session.execute_async(statement, row_values, execution_profile=execution_profile)
        )
        written += 1
    for future in in_flight:
        future.result()
    return written

`iter_remote_parquet_batches()` recorre cada Parquet remoto por lotes (`RecordBatch`), descargando sólo lo que va leyendo, y el cargador recorre ese iterador **una sola vez**: por cada lote, filtra las filas de cada tabla destino, selecciona las columnas en el orden del `INSERT` preparado y las envía con `populate_table()` y el perfil de carga. Los nulos de Arrow se sustituyen por `UNSET_VALUE` para no escribirlos: en Cassandra un nulo explícito se guarda como una *tombstone*, igual que un borrado. En `Posts`, cada lote pasa antes por una operación de PyArrow que recorta el campo `Body` a 80 caracteres (`MAX_CARACTERES_BODY`): ocupa el 96 % de los bytes del *dump* y ninguna consulta de esta sesión lo lee, así que a Cassandra no se envían los ~790 MB de HTML completos.

In [ ]:
from asyncio import Task, create_task
from collections.abc import Iterable
from dataclasses import dataclass
from queue import Queue

import pyarrow.compute as pc
from cassandra.query import UNSET_VALUE, PreparedStatement


@dataclass
class LoadTarget:
    """Tabla destino de una carga: sentencia preparada, columnas y filtro de filas."""

    name: str
    statement: PreparedStatement
    columns: list[str]
    # Par (columna, valor): sólo se cargan las filas cuya columna valga eso.
    row_filter: tuple[str, int] | None = None


def background_import_parquet_batches(
    session: Session,
    batches: Iterable[pa.RecordBatch],
    targets: list[LoadTarget],
    logqueue: Queue[str] | None = None,
    concurrency: int = 100,
    execution_profile: str = LOAD_PROFILE,
) -> Task[dict[str, int]]:
    """Carga en Cassandra los lotes de un Parquet, en un hilo y sin bloquear el cuaderno.

    Recorre `batches` una sola vez y reparte cada lote entre sus tablas destino.
    Recibe los lotes ya preparados y no sabe de qué fichero vienen ni si se han
    transformado por el camino: leer, transformar y cargar quedan separados, y lo
    particular de un fichero (recortar una columna, por ejemplo) se decide en la
    celda que lanza la carga, no aquí. Como `batches` se recorre dentro del hilo,
    si es un iterador perezoso la descarga también ocurre en segundo plano.
    Devuelve una `Task` que da, al terminar, las filas insertadas por tabla.
    """

    def _column_to_bind_values(column: pa.Array) -> list[object]:
        """Convierte una columna Arrow en valores para el INSERT, sin escribir los nulos.

        Enlazar `None` escribe un nulo explícito, y Cassandra lo guarda como una
        *tombstone* de celda: la misma marca que deja un borrado, que cada lectura
        tiene que saltarse hasta que la compactación la elimina. Con `UNSET_VALUE`
        el *driver* no envía esa columna: la celda no se escribe y al leerla sale
        `None` igualmente. Sólo vale para una carga inicial como ésta: para vaciar
        una columna que ya tiene valor sí hay que escribir el nulo.

        La sustitución se hace por columnas y no por filas: `null_count` lo calcula
        Arrow sobre su mapa de validez, sin pasar por Python, así que las columnas
        sin nulos (la clave, entre otras) se devuelven tal cual.
        """
        values: list[object] = column.to_pylist()
        if column.null_count == 0:
            return values
        return [UNSET_VALUE if value is None else value for value in values]

    def _log(message: str) -> None:
        if logqueue is not None:
            logqueue.put_nowait(message)

    def _load() -> dict[str, int]:
        inserted: dict[str, int] = {target.name: 0 for target in targets}
        try:
            for record_batch in batches:
                for target in targets:
                    batch: pa.RecordBatch = record_batch
                    if target.row_filter:
                        column, value = target.row_filter
                        batch = batch.filter(pc.equal(batch.column(column), value))
                    if batch.num_rows == 0:
                        continue
                    # select() garantiza el mismo orden que el INSERT preparado.
                    batch = batch.select(target.columns)
                    # Arrow convierte los nulos (también las fechas) a None; a
                    # diferencia de pandas, aquí nunca aparecen NaT ni NaN.
                    values: list[list[object]] = [
                        _column_to_bind_values(column) for column in batch.columns
                    ]
                    _log(f"{target.name}: enviando {batch.num_rows:,} filas")
                    populate_table(
                        session,
                        target.statement,
                        zip(*values),
                        execution_profile=execution_profile,
                        concurrency=concurrency,
                    )
                    inserted[target.name] += batch.num_rows
                    total: int = inserted[target.name]
                    _log(f"{target.name}: filas insertadas: {total:,}")
        except Exception as exc:
            _log(f"Error durante la carga: {exc!r}")
            raise
        return inserted

    return create_task(to_thread(_load))


def print_log(logqueue: Queue[str]) -> None:
    """Vacía la cola de progreso de una carga."""
    while not logqueue.empty():
        print(logqueue.get_nowait())

In [ ]:
import asyncio

# Las claves son las de la sesión 5. Questions y Answers salen de Posts.parquet y
# se distinguen por PostTypeId; la columna se usa para filtrar aunque no se inserte.
questions_excluded_fields: set[str] = {"PostTypeId", "ParentId"}
answers_excluded_fields: set[str] = {
    "AcceptedAnswerId",
    "AnswerCount",
    "ClosedDate",
    "PostTypeId",
    "Tags",
    "Title",
}

create_table_statements: list[str] = [
    cassandra_create_table_from_pyarrow(
        tags_schema, table_name="Tags", key_form=["Id"], keyspace="stackoverflow"
    ),
    cassandra_create_table_from_pyarrow(
        users_schema, table_name="Users", key_form=["Id"], keyspace="stackoverflow"
    ),
    cassandra_create_table_from_pyarrow(
        posts_schema,
        table_name="Questions",
        key_form=["Id", "CreationDate"],
        exclude_fields=sorted(questions_excluded_fields),
        clustering_order={"CreationDate": "ASC"},
        keyspace="stackoverflow",
    ),
    cassandra_create_table_from_pyarrow(
        posts_schema,
        table_name="Answers",
        key_form=["ParentId", "CreationDate"],
        exclude_fields=sorted(answers_excluded_fields),
        clustering_order={"CreationDate": "ASC"},
        keyspace="stackoverflow",
    ),
]
for create_table_cql in create_table_statements:
    session.execute(create_table_cql)

tags_insert_cql: str
tags_columns: list[str]
tags_insert_cql, tags_columns = cassandra_insert_template(tags_schema, "stackoverflow.Tags")

tags_target: LoadTarget = LoadTarget(
    name="Tags",
    statement=session.prepare(tags_insert_cql),
    columns=tags_columns,
)
users_insert_cql: str
users_columns: list[str]
users_insert_cql, users_columns = cassandra_insert_template(users_schema, "stackoverflow.Users")

users_target: LoadTarget = LoadTarget(
    name="Users",
    statement=session.prepare(users_insert_cql),
    columns=users_columns,
)
questions_insert_cql: str
questions_columns: list[str]
questions_insert_cql, questions_columns = cassandra_insert_template(
    posts_schema, "stackoverflow.Questions", exclude_fields=questions_excluded_fields
)

questions_target: LoadTarget = LoadTarget(
    name="Questions",
    statement=session.prepare(questions_insert_cql),
    columns=questions_columns,
    row_filter=("PostTypeId", 1),
)
answers_insert_cql: str
answers_columns: list[str]
answers_insert_cql, answers_columns = cassandra_insert_template(
    posts_schema, "stackoverflow.Answers", exclude_fields=answers_excluded_fields
)

answers_target: LoadTarget = LoadTarget(
    name="Answers",
    statement=session.prepare(answers_insert_cql),
    columns=answers_columns,
    row_filter=("PostTypeId", 2),
)

# El cuerpo de las publicaciones se guarda recortado a MAX_CARACTERES_BODY
# caracteres.
MAX_CARACTERES_BODY: int = 80

# Tags y Posts se cargan a la vez en segundo plano. Users empieza cuando termina
# Posts: con pocos núcleos, como en Colab, dos cargas a la vez no acaban antes que
# una tras otra, y así Questions y Answers están listas mucho antes.
tags_log_queue: Queue[str] = Queue()
users_log_queue: Queue[str] = Queue()
posts_log_queue: Queue[str] = Queue()
future_tags: Task[dict[str, int]] = background_import_parquet_batches(
    session,
    iter_remote_parquet_batches(f"{PARQUET_BASE_URL}/Tags.parquet"),
    [tags_target],
    logqueue=tags_log_queue,
)
future_posts: Task[dict[str, int]] = background_import_parquet_batches(
    session,
    # El recorte se aplica lote a lote con map(), que es perezoso: aquí no se
    # descarga ni se transforma nada, sino en el hilo de la carga, según se lee
    # cada lote. Es una operación de Arrow sobre la columna entera, sin pasar por
    # objetos de Python. Va escrito dentro de la llamada, y no en una variable,
    # porque un iterador sólo se puede recorrer una vez: así cada ejecución de
    # la carga crea el suyo y no puede reutilizarse uno ya agotado.
    map(
        lambda batch: batch.set_column(
            batch.schema.get_field_index("Body"),
            "Body",
            pc.utf8_slice_codeunits(batch.column("Body"), start=0, stop=MAX_CARACTERES_BODY),
        ),
        iter_remote_parquet_batches(f"{PARQUET_BASE_URL}/Posts.parquet"),
    ),
    [questions_target, answers_target],
    logqueue=posts_log_queue,
)


async def import_users_after_posts() -> dict[str, int]:
    """Espera a que termine la carga de Posts y lanza entonces la de Users."""
    users_log_queue.put_nowait("Users: esperando a que termine la carga de Posts")
    # asyncio.wait() no propaga el error de Posts: ése lo muestra su propia espera.
    await asyncio.wait((future_posts,))
    return await background_import_parquet_batches(
        session,
        iter_remote_parquet_batches(f"{PARQUET_BASE_URL}/Users.parquet"),
        [users_target],
        logqueue=users_log_queue,
    )


future_users: Task[dict[str, int]] = create_task(import_users_after_posts())

## Mientras se cargan los datos: escrituras con otra semántica

Las cargas siguen en segundo plano. `Tags` termina en unos segundos; `Posts` tarda unos minutos y, cuando termina, empieza `Users`, que necesita otros tantos (entre las dos, unos nueve en Google Colab). Se cargan una tras otra porque, con pocos núcleos, como en Colab, dos cargas a la vez no terminan antes, y así `Questions` y `Answers` están listas mucho antes. No hace falta esperarlas para seguir: lo que viene ahora no lee las tablas cargadas, y la espera llega cuando una celda necesita los datos. `Tags` y `Posts` se esperan antes de `GROUP BY`, y `Users`, que sólo usan los ejercicios evaluables, justo antes de ellos.

In [ ]:
# @title Diapositiva: Escrituras con otra semántica
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Escrituras con otra semántica</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>TTL: datos que caducan solos, columna a columna</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>WRITETIME()</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>TTL()</code> para auditar una celda</li><li style=\'margin:10px 0;list-style-position:outside\'>Escrituras condicionales (<code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>IF NOT EXISTS</code>) y lo que cuestan</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>BATCH</code> para atomicidad, no para rendimiento</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Toda escritura es un <em>upsert</em>, salvo que se pida otra cosa</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

### TTL, marca temporal y escritura condicional

En Cassandra toda escritura es un *upsert*: un `INSERT` sobre una clave que ya existe la sobrescribe, sin error. Tres herramientas cambian esa semántica o permiten observarla:

- `USING TTL n`, al final de un `INSERT` o un `UPDATE`, hace que las celdas escritas caduquen a los `n` segundos. El TTL es **por columna**: si después se escribe otra columna sin TTL, la fila puede quedar a medio caducar.
- `TTL(columna)` devuelve los segundos de vida que le quedan a una celda, y `WRITETIME(columna)`, la marca temporal (en microsegundos) con la que se escribió, que es la que usa Cassandra para decidir qué versión gana.
- `IF NOT EXISTS` convierte el `INSERT` en una **escritura condicional** (*lightweight transaction*): sólo se aplica si la fila no existía, y la respuesta trae la columna `[applied]` (`applied` en la fila que devuelve el *driver*) con `True` o `False`. Se coordina con Paxos entre las réplicas, con varias idas y vueltas: es para cuando la condición es necesaria, no el camino ordinario de escritura.

```sql
INSERT INTO t (k, c, v) VALUES (?, ?, ?) IF NOT EXISTS USING TTL 300;
SELECT v, TTL(v), WRITETIME(v) FROM t WHERE k = ? AND c = ?;
```

### `BATCH`: atomicidad, no rendimiento

Pese a lo que sugiere el nombre, un `BATCH` de Cassandra no existe para acelerar varias escrituras, sino para que se apliquen **todas o ninguna**. Hay dos casos:

- Si todas las escrituras caen en **una sola partición** de una tabla, se aplican como una única mutación en las réplicas de esa partición: es atómico y aislado, y no cuesta más que una escritura normal.
- Un `BATCH` *logged* (el tipo por omisión de `BatchStatement`) que abarca **varias particiones** —de una o de varias tablas— se guarda antes en el *batchlog*, que garantiza que todas las escrituras acabarán aplicándose aunque falle el coordinador. Es el uso típico para mantener sincronizada una tabla con sus copias desnormalizadas, pero ese registro previo lo hace más lento que las escrituras sueltas, y no aísla: otro cliente puede ver una parte aplicada y otra todavía no.

El ejemplo siguiente es del segundo tipo: escribe en dos particiones de la misma tabla (las preguntas 1003 y 1004), así que Cassandra usa el *batchlog*. Al final se leen las dos filas.

In [ ]:
from cassandra.query import BatchStatement, SimpleStatement

session.execute("""
    CREATE TABLE IF NOT EXISTS stackoverflow.preguntas_revisadas (
        question_id bigint PRIMARY KEY,
        revisor text
    );
""")
# Crear si no existe y vaciar sólo sirve para que la celda sea reproducible. Con
# DROP + CREATE seguidos, la tabla recreada es otra para Cassandra, y una
# escritura inmediata puede llegar a un nodo que aún no la conoce
# (INCOMPATIBLE_SCHEMA).
session.execute("TRUNCATE stackoverflow.preguntas_revisadas")

batch: BatchStatement = BatchStatement()
batch.add(
    SimpleStatement(
        "INSERT INTO stackoverflow.preguntas_revisadas (question_id, revisor) VALUES (1003, 'ana');"
    )
)
batch.add(
    SimpleStatement(
        "INSERT INTO stackoverflow.preguntas_revisadas (question_id, revisor) VALUES (1004, 'ana');"
    )
)
session.execute(batch)

# Las dos filas se han aplicado juntas (o no se habría aplicado ninguna).
session.execute("SELECT * FROM stackoverflow.preguntas_revisadas").all()

In [ ]:
# @title Diapositiva: Resumen: escrituras, TTL y consistencia
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Resumen: escrituras, TTL y consistencia</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Toda escritura es un <em>upsert</em>: no hay error por «ya existía» salvo con <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>IF NOT EXISTS</code></li><li style=\'margin:10px 0;list-style-position:outside\'>Una escritura condicional usa Paxos: es correcta, pero costosa, y no debe ser el caso normal</li><li style=\'margin:10px 0;list-style-position:outside\'>Un <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>BATCH</code> da atomicidad, no velocidad: sólo es eficiente dentro de una partición</li><li style=\'margin:10px 0;list-style-position:outside\'>El TTL se aplica por columna, así que puede dejar una fila a medio caducar</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>WRITETIME()</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>TTL()</code> permiten auditar cuándo se escribió un dato y cuánto le queda</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>R + W &gt; RF</code> es la condición para leer siempre la última escritura confirmada</li><li style=\'margin:10px 0;list-style-position:outside\'>Un borrado escribe una <em>tombstone</em>: acumularlas degrada las lecturas</li></ul><hr color="#287681" size="2" noshade style="display:none"></div>'))

## Fin de la carga de `Tags` y `Posts`

`GROUP BY` lee `Answers` y el ensayo, `Questions`, así que aquí esperamos a las cargas de `Tags` y de `Posts`. La de `Users` empieza entonces y sigue en segundo plano.

In [ ]:
# Esperar a las cargas de Tags y Posts. La de Users empieza entonces en segundo
# plano: se espera antes de los ejercicios, que son los únicos que la usan.
_done: set[Task[dict[str, int]]]
pending: set[Task[dict[str, int]]]
_done, pending = await asyncio.wait(
    (future_tags, future_posts),
    timeout=1800,
    return_when=asyncio.ALL_COMPLETED,
)

print_log(tags_log_queue)
print_log(posts_log_queue)

if pending:
    raise TimeoutError(f"La carga no terminó a tiempo: quedan {len(pending)} tareas pendientes.")

# Al recoger el resultado se propagan los errores ocurridos en los hilos.
tags_inserted: dict[str, int] = await future_tags
posts_inserted: dict[str, int] = await future_posts
print(tags_inserted, posts_inserted)

### Comprobación: ¿han llegado todas las filas?

La carga filtra `Posts` por `PostTypeId`, descarta columnas y no escribe los nulos. Contrastamos el **número de filas** en cada etapa: las que se esperan del dataset (cifras conocidas, que se actualizan aquí si los datos cambian), las que ha enviado el cargador y las que hay en Cassandra, contadas con un `COUNT(*)` por cada trozo del anillo de tokens, en paralelo. La última cifra importa porque en Cassandra un `INSERT` sobre una clave que ya existe es una actualización, no un error: si dos respuestas de la misma pregunta tuvieran el mismo `CreationDate`, la segunda sustituiría en silencio a la primera. `Users` se comprueba igual cuando termine su carga, antes de los ejercicios.

In [ ]:
from cassandra.concurrent import execute_concurrent_with_args

# Filas que debe tener cada tabla con el dataset 26-27.
EXPECTED_TAGS: int = 2_996
EXPECTED_QUESTIONS: int = 197_149
EXPECTED_ANSWERS: int = 218_679


# Cassandra: un COUNT(*) de toda la tabla tiene un único plazo para todas sus páginas
# internas (range_request_timeout, 10 s) y con estas tablas puede agotarlo (ReadTimeout).
# Contar las filas en el cliente evita el plazo, pero obliga a traerlas todas. Aquí se
# reparte el anillo de tokens en trozos: cada COUNT(*) cuenta uno, con su propio plazo,
# todos a la vez, y el servidor devuelve sólo un número por trozo.
def count_rows_by_token_range(
    session: Session,
    table: str,
    partition_key: str,
    ranges: int = 64,
    concurrency: int = 32,
) -> int:
    """Cuenta las filas de una tabla con un `COUNT(*)` por cada trozo del anillo de tokens."""
    # El anillo del Murmur3Partitioner, el de Cassandra por omisión, va de -2^63 a 2^63 - 1.
    min_token: int = -(2**63)
    max_token: int = 2**63 - 1
    width: int = (max_token - min_token) // ranges + 1
    bounds: list[tuple[int, int]] = [
        (min_token + i * width, min(min_token + (i + 1) * width - 1, max_token))
        for i in range(ranges)
    ]
    count_statement: PreparedStatement = session.prepare(
        f"SELECT COUNT(*) FROM {table} "
        f"WHERE token({partition_key}) >= ? AND token({partition_key}) <= ?"
    )
    # `bounds` es una lista pequeña, no un generador perezoso: no hay riesgo de colgar
    # el hilo de red del driver. Si un trozo falla, lanza su excepción.
    results: list[tuple[bool, Any]] = execute_concurrent_with_args(
        session, count_statement, bounds, concurrency=concurrency
    )
    return sum(result.one()[0] for _, result in results)


tags_in_cassandra: int = count_rows_by_token_range(session, "stackoverflow.Tags", "Id")
questions_in_cassandra: int = count_rows_by_token_range(session, "stackoverflow.Questions", "Id")
answers_in_cassandra: int = count_rows_by_token_range(session, "stackoverflow.Answers", "ParentId")

# Lo que ha enviado el cargador, tal como lo devolvieron las tareas
print(
    f"Tags       Esperadas {EXPECTED_TAGS:>8,} | enviadas {tags_inserted['Tags']:>8,} | Cassandra {tags_in_cassandra:>8,}"
)
print(
    f"Questions  Esperadas {EXPECTED_QUESTIONS:>8,} | enviadas {posts_inserted['Questions']:>8,} | Cassandra {questions_in_cassandra:>8,}"
)
print(
    f"Answers    Esperadas {EXPECTED_ANSWERS:>8,} | enviadas {posts_inserted['Answers']:>8,} | Cassandra {answers_in_cassandra:>8,}"
)

assert EXPECTED_TAGS == tags_inserted["Tags"] == tags_in_cassandra, "Cambian las filas de Tags"
assert EXPECTED_QUESTIONS == posts_inserted["Questions"] == questions_in_cassandra, (
    "Cambian las filas de Questions"
)
assert EXPECTED_ANSWERS == posts_inserted["Answers"] == answers_in_cassandra, (
    "Cambian las filas de Answers: ¿dos respuestas con la misma clave (ParentId, CreationDate)?"
)
print("Las filas de Tags, Questions y Answers coinciden en las tres etapas")

In [ ]:
# @title Diapositiva: La base de datos está construida
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">La base de datos está construida</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Cuatro tablas, una por consulta: <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Tags</code>, <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Users</code>, <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Questions</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Answers</code></li><li style=\'margin:10px 0;list-style-position:outside\'>Las cargas van en segundo plano, <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Users</code> tras <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Posts</code>, y cada tabla se espera cuando hace falta</li><li style=\'margin:10px 0;list-style-position:outside\'>Cada tabla se contrasta por filas: esperadas, cargador y Cassandra</li><li style=\'margin:10px 0;list-style-position:outside\'>Las claves son las de la sesión 5, con sus límites: sin JOIN y sin rangos fuera de la partición</li><li style=\'margin:10px 0;list-style-position:outside\'>Aquí construimos sobre ellas: tablas derivadas, agregaciones y escrituras</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Con los datos cargados, la sesión se dedica a lo que se añade a la clave</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: GROUP BY: agregar dentro de la clave
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px"><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>GROUP BY</code>: agregar dentro de la clave</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Agrupar por la clave de partición: un grupo por partición</li><li style=\'margin:10px 0;list-style-position:outside\'>Ver qué columnas admite <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>GROUP BY</code> y por qué</li><li style=\'margin:10px 0;list-style-position:outside\'>Saber dónde se agrega lo que <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>GROUP BY</code> no puede</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Cassandra agrega dentro de una partición; el resto se calcula fuera</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

## `GROUP BY`: sólo dentro de la clave

`GROUP BY` sólo admite columnas de la clave primaria y en su orden: primero la clave de partición y, después, las de *clustering*. Por eso un grupo nunca mezcla filas de dos particiones, y no existe un `GROUP BY` por una columna cualquiera, como en SQL. En `Answers` (clave `(ParentId, CreationDate)`), agrupar por `ParentId` da una fila por pregunta con el número de sus respuestas y la media de su puntuación; agrupar por `ParentId, CreationDate` daría casi siempre un grupo por respuesta, y por `CreationDate` sola Cassandra lo rechaza, porque falta la clave de partición. Una agregación sobre toda la tabla, como la media de todo el foro, no se hace en Cassandra: se calcula fuera (en la aplicación o con Spark) o se mantiene en una tabla de resumen escrita a la vez que los datos, como las de los ejercicios. Además, `AVG` devuelve el tipo de la columna: sobre un entero como `Score`, la media se trunca.


In [ ]:
session.execute("""
    -- Un grupo por partición (pregunta). Sin WHERE se recorren las particiones en
    -- orden de token; LIMIT corta tras cinco grupos.
    SELECT ParentId, COUNT(*) AS NumAnswers, AVG(Score) AS AverageScore
    FROM stackoverflow.Answers
    GROUP BY ParentId
    LIMIT 5;
""").all()

In [ ]:
# @title Diapositiva: Para el examen: GROUP BY en Cassandra
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #8a6a00;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#8a6a00" size="8" noshade style="display:none"><div style="color:#8a6a00;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#8a6a00" style="color:inherit"><b>Para el examen</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Para el examen: <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>GROUP BY</code> en Cassandra</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>GROUP BY</code> sólo admite columnas de la clave primaria, en su orden: primero la partición y después el <em>clustering</em></li><li style=\'margin:10px 0;list-style-position:outside\'>Un grupo nunca mezcla particiones: no hay <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>GROUP BY</code> por una columna cualquiera, como en SQL</li><li style=\'margin:10px 0;list-style-position:outside\'>Agregar sobre toda la tabla se hace fuera (aplicación o Spark) o con una tabla de resumen escrita a la vez que los datos</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>AVG</code> devuelve el tipo de la columna: sobre un entero, la media se trunca</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #8a6a00;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Pregunta típica: explicar por qué una agregación no se puede hacer con <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>GROUP BY</code> y dónde se hace</div></blockquote><hr color="#8a6a00" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: ¿Qué tabla sirve a esta consulta?
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #a33a2b;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#a33a2b" size="8" noshade style="display:none"><div style="color:#a33a2b;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#a33a2b" style="color:inherit"><b>Pregunta guía</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">¿Qué tabla sirve a esta consulta?</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Consulta: las 10 preguntas sin respuestas más recientes de una etiqueta</li><li style=\'margin:10px 0;list-style-position:outside\'>¿Qué columna es la clave de partición? ¿Y el <em>clustering</em>, con qué orden?</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>Tags</code> es una cadena: una pregunta con tres etiquetas, ¿cuántas filas genera?</li><li style=\'margin:10px 0;list-style-position:outside\'>¿Cuántas filas tendrá la partición de la etiqueta más frecuente, y qué pasa cuando crece?</li><li style=\'margin:10px 0;list-style-position:outside\'>¿Se puede contestar sin ordenar en el cliente ni usar <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>ALLOW FILTERING</code>?</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #a33a2b;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Primero se decide la clave en clase; después se ejecuta y se comprueba</div></blockquote><hr color="#a33a2b" size="2" noshade style="display:none"></div>'))

## Ensayo antes de los ejercicios: preguntas sin respuesta por etiqueta

Antes de ejecutar la celda siguiente, decidid en clase la `PRIMARY KEY` de una tabla que conteste a:
> **las 10 preguntas sin respuestas (`AnswerCount` a 0 o sin valor) más recientes de una etiqueta**.

Se trabaja igual que en los ejercicios evaluables, con otra pregunta: se recorre `Questions` una vez en el cliente, se separan las etiquetas de `Tags` y se escribe una tabla nueva con `populate_table()`.

La tabla `unanswered_by_tag_demo` tiene una fila por pregunta sin respuestas y etiqueta, con las columnas `tag` (una etiqueta de la pregunta), `created_at` (`Questions.CreationDate`), `question_id` (`Questions.Id`) y `title` (`Questions.Title`). Se lee justo después de escribirla, así que se escribe con el perfil por omisión, no con el de carga.

In [ ]:
from collections import Counter
from datetime import datetime

# Crear si no existe y vaciar, como en el BATCH: sólo para poder reejecutar la celda.
session.execute("""
    CREATE TABLE IF NOT EXISTS stackoverflow.unanswered_by_tag_demo (
        tag text,
        created_at timestamp,
        question_id bigint,
        title text,
        PRIMARY KEY ((tag), created_at, question_id)
    ) WITH CLUSTERING ORDER BY (created_at DESC, question_id ASC)
""")
session.execute("TRUNCATE stackoverflow.unanswered_by_tag_demo")
insert_unanswered: PreparedStatement = session.prepare("""
    INSERT INTO stackoverflow.unanswered_by_tag_demo (tag, created_at, question_id, title)
    VALUES (?, ?, ?, ?)
""")

# Una pregunta sin respuestas genera una fila por cada una de sus etiquetas.
unanswered_rows: list[tuple[str, datetime, int, str]] = [
    (tag, question.creationdate, question.id, question.title)
    for question in session.execute(
        "SELECT Id, CreationDate, Title, Tags, AnswerCount FROM stackoverflow.Questions"
    )
    if not question.answercount
    for tag in (question.tags or "").strip("<>").split("><")
    if tag
]
# La lectura siguiente depende de esta escritura: perfil por omisión, no el de carga.
# Fijaos que si no hubiera que cambiar o filtrar las filas de la tabla origen (Questions),
# no haría falta crear el array unanswered_rows, sino que se podría pasar directamente
# el ResultSet a populate_table().
unanswered_written: int = populate_table(
    session, insert_unanswered, unanswered_rows, execution_profile=EXEC_PROFILE_DEFAULT
)

partition_sizes: Counter[str] = Counter(row[0] for row in unanswered_rows)
busiest_tag: str
busiest_rows: int
busiest_tag, busiest_rows = partition_sizes.most_common(1)[0]
print(
    f"Filas escritas: {unanswered_written:,} en {len(partition_sizes):,} particiones "
    f"(etiquetas); la mayor, <{busiest_tag}>, con {busiest_rows:,} filas"
)

In [ ]:
# Una sola partición, ya ordenada por el clustering: ni ORDER BY ni filtrado.
for row in session.execute(
    "SELECT created_at, question_id, title FROM stackoverflow.unanswered_by_tag_demo "
    "WHERE tag = %s LIMIT 10",
    (busiest_tag,),
):
    print(row.created_at, row.question_id, row.title)

La consulta se resuelve leyendo **una sola partición**, y como `created_at DESC` es el primer *clustering*, las 10 más recientes son las primeras filas: no hay que ordenar en el cliente. Lo que se paga es la escritura: cada pregunta se guarda una vez por etiqueta.

Fijaos ahora en el tamaño de la mayor partición que ha impreso la celda. La partición de una etiqueta popular **crece sin límite** con los años, aunque esta consulta sólo lea el principio. Cambiar la clave de partición a `(tag, year_month)` la acota, como hace el EJERCICIO 2, pero entonces las 10 más recientes pueden repartirse entre varias particiones y la aplicación tiene que decidir cuántas leer. Es el compromiso típico entre partición acotada y consulta sencilla.

## La carga de `Users`

Los ejercicios evaluables leen también `Users`. Esperamos ahora a su carga, si no ha terminado ya, y la comprobamos como las demás: filas esperadas, filas enviadas y filas en Cassandra.

In [ ]:
# Esperar a la carga de Users, la única que puede seguir en marcha.
_done: set[Task[dict[str, int]]]
pending: set[Task[dict[str, int]]]
_done, pending = await asyncio.wait((future_users,), timeout=1800)

print_log(users_log_queue)

if pending:
    raise TimeoutError("La carga de Users no terminó a tiempo.")

# Al recoger el resultado se propagan los errores ocurridos en el hilo.
users_inserted: dict[str, int] = await future_users
print(users_inserted)

In [ ]:
# Filas que debe tener Users con el dataset 26-27; si cambia, se actualiza aquí.
EXPECTED_USERS: int = 469_417

# Cassandra: contadas por trozos del anillo de tokens, como las demás tablas grandes
users_in_cassandra: int = count_rows_by_token_range(session, "stackoverflow.Users", "Id")

# Lo que ha enviado el cargador, tal como lo devolvió la tarea
print(
    f"Users      Esperadas {EXPECTED_USERS:>8,} | enviadas {users_inserted['Users']:>8,} | Cassandra {users_in_cassandra:>8,}"
)

assert EXPECTED_USERS == users_inserted["Users"] == users_in_cassandra, "Cambian las filas de Users"
print("Las filas de Users coinciden en las tres etapas")

In [ ]:
# @title Diapositiva: De la tabla al sistema de acceso
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">De la tabla al sistema de acceso</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Las tablas derivadas sustituyen a los JOIN que Cassandra no ejecuta</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>GROUP BY</code> sólo agrupa dentro de una partición: lo demás se agrega fuera o en una tabla de resumen</li><li style=\'margin:10px 0;list-style-position:outside\'>La clave de partición acota el tamaño de la partición y decide qué consultas son eficientes</li><li style=\'margin:10px 0;list-style-position:outside\'>TTL y LWT cambian la semántica de las escrituras, y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>BATCH</code> sólo da atomicidad</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>La solución completa combina el diseño de las claves y la semántica de las escrituras</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: El boletín evaluable
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">El boletín evaluable</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Los ejercicios se hacen en orden: varios consumen las tablas que dejaron los anteriores</li><li style=\'margin:10px 0;list-style-position:outside\'>Los parámetros con los que trabajas salen de tu identificador</li><li style=\'margin:10px 0;list-style-position:outside\'>El resultado final es pequeño, ordenado y se lee de una sola partición</li><li style=\'margin:10px 0;list-style-position:outside\'>Los ejercicios terminan respondiendo a la pregunta «a quién preguntar según el tema (tag)» en un año concreto</li><li style=\'margin:10px 0;list-style-position:outside\'>Se entrega código y resultado</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Hacer los ejercicios en orden deja preparadas las estructuras del resultado final</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

# Ejercicios evaluables

El boletín contiene una serie de **ejercicios evaluables**.

De cada ejercicio se requiere: **(a)** código y resultado; y **(b)** cuando se solicite, una explicación del diseño o del resultado.

Cuando un enunciado pida explicar o interpretar algo, escríbelo como comentario en la propia celda de la solución, antes o después del código. No se envía con la entrega, pero es lo que tendrás que defender sin el cuaderno.

## Preparación de la entrega

Los ejercicios evaluables y la entrega dependen del identificador que escribiste al principio del cuaderno, en «Identificación de la entrega». La celda siguiente te lo vuelve a mostrar: **compruébalo de nuevo** antes de seguir. De él salen tus parámetros y la huella con la que se registra el resultado, así que un identificador equivocado obliga a repetir los ejercicios. Si no es el tuyo, corrígelo en aquella celda, vuelve a ejecutarla y continúa desde aquí.

In [ ]:
print(f"El DNI que has seleccionado para la entrega es: {DNI}")
print("Compruébalo de nuevo antes de continuar.")

Las dos celdas siguientes preparan lo que usa el resto del boletín. La primera descarga el cliente de entregas del repositorio público `dsevilla/bdge`, así que no hay que copiar ningún ZIP junto al cuaderno. La segunda calcula con él tu huella. Ninguna de las dos envía nada: el envío es el último paso, cuando hayas terminado los ejercicios.

In [ ]:
import importlib.util
import json
import pathlib
import re
import sys
import urllib.error
import urllib.request
import zipfile
from types import ModuleType

CLIENTE_BASE_URL: str = (
    "https://raw.githubusercontent.com/dsevilla/bdge/26-27/addendum/practice-evidence/dist"
)


def carga_cliente_evidencias() -> ModuleType:
    """Consulta la versión publicada, descarga el cliente para este intérprete y lo importa."""
    if "bdge_evidence" in sys.modules:
        return sys.modules["bdge_evidence"]

    cache_tag: str = sys.implementation.cache_tag
    manifiesto_url: str = f"{CLIENTE_BASE_URL}/manifest.json"
    try:
        with urllib.request.urlopen(manifiesto_url, timeout=30) as respuesta:
            manifiesto = json.loads(respuesta.read().decode("utf-8"))
    except (urllib.error.URLError, TimeoutError, ValueError) as error:
        raise RuntimeError(
            f"No se pudo consultar la versión del cliente en {manifiesto_url}"
        ) from error

    if not isinstance(manifiesto, dict) or manifiesto.get("format") != "bdge-evidence-release/v1":
        raise RuntimeError(
            f"El manifiesto del cliente no tiene un formato reconocido: {manifiesto_url}"
        )
    version_cliente = manifiesto.get("client_release")
    if not isinstance(version_cliente, str) or not re.fullmatch(
        r"[0-9]+\.[0-9]+\.[0-9]+", version_cliente
    ):
        raise RuntimeError(f"El manifiesto no contiene una versión válida: {manifiesto_url}")

    nombre: str = f"bdge-evidence-{version_cliente}-{cache_tag}.zip"
    destino: pathlib.Path = pathlib.Path.cwd() / nombre
    if not destino.exists():
        url: str = f"{CLIENTE_BASE_URL}/{nombre}"
        try:
            with urllib.request.urlopen(url, timeout=30) as respuesta:
                destino.write_bytes(respuesta.read())
        except urllib.error.HTTPError as error:
            raise RuntimeError(
                f"No hay cliente publicado para {cache_tag}: {url} ({error.code})"
            ) from error
        print(f"Cliente descargado: {nombre}")

    # El ZIP lleva bytecode y manifiesto, pero debe coincidir con este Python.
    with zipfile.ZipFile(destino) as archivo:
        magico: bytes = archivo.read("bdge_evidence.pyc")[:4]
        manifiesto_zip = json.loads(archivo.read("manifest.json").decode("utf-8"))
    if magico != importlib.util.MAGIC_NUMBER:
        raise RuntimeError(
            f"{nombre} se compiló con otro Python y no puede cargarse en este "
            f"kernel ({sys.version.split()[0]}, {cache_tag}). Borra el fichero "
            "y vuelve a ejecutar esta celda."
        )
    if (
        not isinstance(manifiesto_zip, dict)
        or manifiesto_zip.get("client_version") != f"bdge-evidence-demo/{version_cliente}"
    ):
        raise RuntimeError(f"El ZIP no corresponde a la versión publicada: {nombre}")

    sys.path.insert(0, str(destino))
    import bdge_evidence

    return bdge_evidence


evidence: ModuleType = carga_cliente_evidencias()
print(f"Cliente de evidencias {evidence.CLIENT_VERSION} disponible")

In [ ]:
HUELLA: str = evidence.hash_student_identifier(DNI, CURSO, BOLETIN)

if evidence.is_example_identifier(DNI):
    print("AVISO: sigues usando el identificador de ejemplo. Cámbialo antes de entregar.")
print(f"Curso: {CURSO} | Boletín: {BOLETIN}")
print(f"Huella del alumno (esto es lo único que se registra): {HUELLA}")

## Parámetros personales del boletín

El DNI no selecciona un caso de una lista: sus dígitos entran directamente
en las consultas. Los rangos están acotados para que todos trabajéis con un
volumen razonable. La letra final no interviene ni en los cálculos ni en la
huella que identifica la entrega.


In [ ]:
from dataclasses import dataclass
from pprint import pprint


@dataclass(frozen=True)
class ParametrosBoletin:
    numero: int
    umbral_reputacion: int
    año_inicial: int
    mes: int
    minimo_actividad: int
    limite: int
    factor_a: int
    factor_b: int


NUMERO_DNI: int = int(evidence.normalize_student_identifier(DNI))
PARAMETROS: ParametrosBoletin = ParametrosBoletin(
    numero=NUMERO_DNI,
    umbral_reputacion=100 + NUMERO_DNI % 4_900,
    año_inicial=2016 + (NUMERO_DNI // 10) % 5,
    mes=1 + (NUMERO_DNI // 100) % 12,
    minimo_actividad=2 + (NUMERO_DNI // 1_000) % 4,
    limite=5,
    factor_a=2 + NUMERO_DNI % 5,
    factor_b=3 + (NUMERO_DNI // 10) % 7,
)
pprint(PARAMETROS)

### Tu año

La cadena de ejercicios estudia **un año completo del foro**: el año natural `PARAMETROS.año_inicial`, del 1 de enero al 31 de diciembre. Con el año 2017, por ejemplo, va de enero a diciembre de 2017.

> En general, el hilo de los ejercicios es terminar consultando  «a quién preguntar según el tema (tag)», y cada ejercicio de la cadena prepara una pieza.

De `PARAMETROS`, este boletín usa `umbral_reputacion`, `año_inicial`, `minimo_actividad` y `limite`, y el EJERCICIO 3 usa además `mes`. `factor_a` y `factor_b` no intervienen: se derivan igual en todos los boletines, pero aquí no se usan.

La celda siguiente fija los límites de tu año: úsalos en los ejercicios.

In [ ]:
from datetime import datetime

# El año incluye INICIO_AÑO y excluye FIN_AÑO: un año natural completo. El
# driver devuelve las fechas de Cassandra como `datetime` sin zona horaria (UTC).
INICIO_AÑO: datetime = datetime(PARAMETROS.año_inicial, 1, 1)
FIN_AÑO: datetime = datetime(PARAMETROS.año_inicial + 1, 1, 1)
print(f"Tu año: desde el {INICIO_AÑO:%Y-%m-%d} hasta el {FIN_AÑO:%Y-%m-%d}, este último excluido")

### EJERCICIO 1: respondedores habituales

Para que el cuaderno se pueda reejecutar entero, empieza con
`DROP TABLE IF EXISTS b6_regular_answerers`. En los ejercicios se borra la tabla, y no se vacía con
`CREATE TABLE IF NOT EXISTS` y `TRUNCATE` como en el `BATCH`, porque puedes cambiar la clave de un
intento a otro, y `IF NOT EXISTS` conservaría sin avisar la definición anterior. El riesgo que señala
el `BATCH` (escribir en una tabla que algún nodo aún no conoce) lo cubre la conexión, que tras cada
cambio de esquema espera hasta 30 s a que todos los nodos lo tengan (`max_schema_agreement_wait`).

Recorre `Users` (una fila por usuario) y `Answers` (una fila por respuesta) una sola vez en el cliente (puedes utilizar memoria para almacenar información temporal). Los **respondedores habituales** son los usuarios con reputación (`Users.Reputation`) de al menos `PARAMETROS.umbral_reputacion` que escribieron al menos `PARAMETROS.minimo_actividad` respuestas dentro de tu año (desde `INICIO_AÑO`, incluida, hasta `FIN_AÑO`, excluida), contando todas sus respuestas de ese año, estén o no aceptadas. Las respuestas sin autor (`Answers.OwnerUserId` vacío) no cuentan para nadie.

Crea `b6_regular_answerers`, con **una fila por respondedor habitual** y clave `((answerer_id))`. Sus columnas son:

- `answerer_id`: el usuario, `Users.Id`;
- `reputation`: su reputación, `Users.Reputation`;
- `answers_in_year`: cuántas respuestas escribió en tu año.

Usa una sentencia preparada y `populate_table()` con `execution_profile=EXEC_PROFILE_DEFAULT`, el perfil por omisión: el EJERCICIO 2 leerá enseguida lo que escribas, y la consistencia `ANY` del perfil de carga no garantiza esa lectura. Lo mismo vale para las tablas de los ejercicios 2, 4, 6 y 7.

In [ ]:
# Comprobación del EJERCICIO 1: b6_regular_answerers
habituales: list[Any] = session.execute(
    "SELECT answerer_id, reputation, answers_in_year FROM stackoverflow.b6_regular_answerers"
).all()
assert habituales, "b6_regular_answerers está vacía"

# Lo que exige la definición, en todas las filas de la tabla.
assert all(fila.reputation >= PARAMETROS.umbral_reputacion for fila in habituales), (
    "Hay respondedores por debajo del umbral de reputación"
)
assert all(fila.answers_in_year >= PARAMETROS.minimo_actividad for fila in habituales), (
    "Hay respondedores por debajo de la actividad mínima"
)

# Camino paralelo: una muestra con los menos activos (los que más cerca están del
# mínimo) y las primeras filas se contrasta con Users en consultas puntuales.
por_actividad: list[Any] = sorted(habituales, key=lambda f: (f.answers_in_year, f.answerer_id))
menos_activos: list[Any] = por_actividad[:3]
verificados: list[Any] = menos_activos + habituales[:12]
for fila in verificados:
    usuario: Any = session.execute(
        "SELECT Reputation FROM stackoverflow.Users WHERE Id = %s", (fila.answerer_id,)
    ).one()
    assert usuario is not None, f"Respondedor {fila.answerer_id} no existe en Users"
    assert usuario.reputation == fila.reputation, (
        f"Respondedor {fila.answerer_id}: reputation no coincide con Users"
    )

# El servidor filtra las respuestas del año de los menos activos. LIMIT pide una
# más de las que declara la tabla: si llegan todas, la tabla cuenta de menos (por
# ejemplo, sólo las aceptadas), y si no llegan a las declaradas, cuenta de más.
for fila in menos_activos:
    respuestas_del_año: int = len(
        session.execute(
            "SELECT Id FROM stackoverflow.Answers WHERE OwnerUserId = %s "
            "AND CreationDate >= %s AND CreationDate < %s LIMIT %s ALLOW FILTERING",
            (fila.answerer_id, INICIO_AÑO, FIN_AÑO, fila.answers_in_year + 1),
        ).all()
    )
    assert respuestas_del_año == fila.answers_in_year, (
        f"Respondedor {fila.answerer_id}: answers_in_year no coincide con sus respuestas del año"
    )
print(
    f"Respondedores habituales: {len(habituales):,} | verificados {len(verificados)} "
    "contra Users/Answers"
)
print("EJERCICIO 1: comprobado")

### EJERCICIO 2: respuestas aceptadas por etiqueta

Para que el cuaderno se pueda reejecutar entero, empieza con
`DROP TABLE IF EXISTS b6_accepted_answers_by_tag`.

Lee los respondedores habituales de `b6_regular_answerers` (EJERCICIO 1) y recorre `Answers` (una fila por respuesta) y `Questions` (una fila por pregunta) una sola vez en el cliente. Una respuesta es **aceptada** cuando su `Id` es el `AcceptedAnswerId` de su pregunta (`Answers.ParentId`).

Crea `b6_accepted_answers_by_tag`, con **una fila por cada respuesta aceptada y etiqueta de su pregunta**, entre las respuestas de los respondedores habituales escritas dentro de tu año (desde `INICIO_AÑO`, incluida, hasta `FIN_AÑO`, excluida). En `Questions`, `Tags` es una cadena como `<python><cassandra>`: sepárala en etiquetas con `Tags.strip("<>").split("><")`. Una pregunta con varias etiquetas genera una fila por etiqueta y una pregunta sin etiquetas no genera ninguna. La clave es `((tag, year_month), answered_at, answer_id)` y el orden de *clustering* `answered_at DESC, answer_id ASC`. Sus columnas son:

- `tag`: una etiqueta de la pregunta respondida;
- `year_month`: el mes de la respuesta con formato `YYYY-MM`, sacado de `Answers.CreationDate`;
- `answered_at`: `Answers.CreationDate`;
- `answer_id`: `Answers.Id`;
- `question_id`: `Answers.ParentId`, la pregunta que se respondió;
- `answerer_id`: el respondedor, `Answers.OwnerUserId`;
- `answer_score`: la puntuación de la respuesta, `Answers.Score` (no la de la pregunta), con 0 si no tiene valor;
- `minutes_to_answer`: los minutos enteros, redondeados hacia abajo, entre `Questions.CreationDate` y `Answers.CreationDate`.

Usa una sentencia preparada y `populate_table()` con `execution_profile=EXEC_PROFILE_DEFAULT`, como en el EJERCICIO 1: el EJERCICIO 4 leerá enseguida lo que escribas.


In [ ]:
# Comprobación del EJERCICIO 2: b6_accepted_answers_by_tag
# Contadas en el cliente, como las tablas cargadas: un COUNT(*) de toda la tabla
# tiene un único plazo para todas sus páginas internas.
total: int = sum(
    1 for _ in session.execute("SELECT answer_id FROM stackoverflow.b6_accepted_answers_by_tag")
)
assert total > 0, "b6_accepted_answers_by_tag está vacía"

# La clave que pide el enunciado, leída del catálogo de esquema: posición de cada
# columna en la clave de partición o de clustering, y su orden.
columnas_clave: dict[str, tuple[str, int, str]] = {
    fila.column_name: (fila.kind, fila.position, fila.clustering_order)
    for fila in session.execute(
        "SELECT column_name, kind, position, clustering_order FROM system_schema.columns "
        "WHERE keyspace_name = 'stackoverflow' AND table_name = 'b6_accepted_answers_by_tag'"
    )
    if fila.kind != "regular"
}
print(f"Columnas de la clave: {columnas_clave}")
assert columnas_clave == {
    "tag": ("partition_key", 0, "none"),
    "year_month": ("partition_key", 1, "none"),
    "answered_at": ("clustering", 0, "desc"),
    "answer_id": ("clustering", 1, "asc"),
}, (
    "La clave debe ser ((tag, year_month), answered_at, answer_id), "
    "con answered_at DESC y answer_id ASC"
)

# Una fila de cada una de las primeras particiones, para que la muestra tenga
# etiquetas y meses distintos.
particiones: list[Any] = session.execute(
    "SELECT DISTINCT tag, year_month FROM stackoverflow.b6_accepted_answers_by_tag LIMIT 15"
).all()
muestra: list[Any] = [
    session.execute(
        "SELECT tag, year_month, answered_at, answer_id, question_id, answerer_id, "
        "answer_score, minutes_to_answer FROM stackoverflow.b6_accepted_answers_by_tag "
        "WHERE tag = %s AND year_month = %s LIMIT 1",
        (particion.tag, particion.year_month),
    ).one()
    for particion in particiones
]

for fila in muestra:
    assert INICIO_AÑO <= fila.answered_at < FIN_AÑO, f"Respuesta {fila.answer_id}: no es de tu año"
    assert fila.year_month == f"{fila.answered_at:%Y-%m}", (
        f"Respuesta {fila.answer_id}: year_month no coincide con answered_at"
    )

    pregunta: Any = session.execute(
        "SELECT CreationDate, Tags, AcceptedAnswerId FROM stackoverflow.Questions WHERE Id = %s",
        (fila.question_id,),
    ).one()
    assert pregunta is not None, f"Pregunta {fila.question_id} no existe en Questions"
    assert pregunta.acceptedanswerid == fila.answer_id, (
        f"Respuesta {fila.answer_id}: no es la aceptada de su pregunta"
    )
    etiquetas: list[str] = [
        etiqueta for etiqueta in (pregunta.tags or "").strip("<>").split("><") if etiqueta
    ]
    assert fila.tag in etiquetas, (
        f"Respuesta {fila.answer_id}: la etiqueta {fila.tag} no es de su pregunta"
    )
    # Una fila por etiqueta: cada etiqueta de la pregunta tiene la suya, con la misma
    # respuesta, leída por su clave primaria completa.
    for etiqueta in etiquetas:
        de_la_etiqueta: Any = session.execute(
            "SELECT answer_id FROM stackoverflow.b6_accepted_answers_by_tag "
            "WHERE tag = %s AND year_month = %s AND answered_at = %s AND answer_id = %s",
            (etiqueta, fila.year_month, fila.answered_at, fila.answer_id),
        ).one()
        assert de_la_etiqueta is not None, (
            f"Respuesta {fila.answer_id}: falta su fila de la etiqueta {etiqueta}"
        )
    minutos: int = int((fila.answered_at - pregunta.creationdate).total_seconds() // 60)
    assert fila.minutes_to_answer == minutos, (
        f"Respuesta {fila.answer_id}: minutes_to_answer no coincide"
    )

    respuestas_de_la_pregunta: list[Any] = session.execute(
        "SELECT Id, OwnerUserId, Score FROM stackoverflow.Answers WHERE ParentId = %s",
        (fila.question_id,),
    ).all()
    respuesta: Any = next((r for r in respuestas_de_la_pregunta if r.id == fila.answer_id), None)
    assert respuesta is not None, (
        f"Respuesta {fila.answer_id}: no está entre las respuestas de su pregunta en Answers"
    )
    assert respuesta.owneruserid == fila.answerer_id, (
        f"Respuesta {fila.answer_id}: respondedor no coincide"
    )
    assert (respuesta.score or 0) == fila.answer_score, (
        f"Respuesta {fila.answer_id}: answer_score no coincide"
    )

    habitual: Any = session.execute(
        "SELECT answerer_id FROM stackoverflow.b6_regular_answerers WHERE answerer_id = %s",
        (fila.answerer_id,),
    ).one()
    assert habitual is not None, (
        f"Respondedor {fila.answerer_id}: no es un respondedor habitual (no está en b6_regular_answerers)"
    )

print(
    f"Filas en b6_accepted_answers_by_tag: {total:,} | verificadas {len(muestra)}, "
    "de particiones distintas, contra b6_regular_answerers/Questions/Answers"
)
print("EJERCICIO 2: comprobado")

### EJERCICIO 3: la consulta que la clave no puede servir

Intenta obtener globalmente todas las respuestas aceptadas del mes personal
(`f"{PARAMETROS.año_inicial}-{PARAMETROS.mes:02d}"`), de cualquier etiqueta, desde la tabla del EJERCICIO 2,
primero sin y luego con `ALLOW FILTERING`. Captura la excepción del primer
intento con `try`/`except Exception as exc` y guarda `type(exc).__name__` en
`clase_error_sin_filtering`; guarda el número de filas del segundo intento en
`filas_con_allow_filtering`. Explica en un comentario de la celda cuántas
particiones obliga a visitar, y diseña en papel la tabla que necesitaría esa
consulta. No crees esa tabla.


In [ ]:
# Comprobación del EJERCICIO 3: la consulta que la clave no puede servir
assert "clase_error_sin_filtering" in globals(), (
    "Guarda el nombre de la excepción en clase_error_sin_filtering"
)
assert "filas_con_allow_filtering" in globals(), "Guarda el número de filas con ALLOW FILTERING"

print(f"Excepción capturada: {clase_error_sin_filtering}")
assert clase_error_sin_filtering == "InvalidRequest", (
    "Sin ALLOW FILTERING, Cassandra debe rechazar la consulta con InvalidRequest"
)

# Camino paralelo: contar en el cliente las filas del mes personal, leyendo
# la tabla completa sin filtrar en el servidor.
mes_personal: str = f"{PARAMETROS.año_inicial}-{PARAMETROS.mes:02d}"
filas_mes_personal: int = sum(
    1
    for fila in session.execute("SELECT year_month FROM stackoverflow.b6_accepted_answers_by_tag")
    if fila.year_month == mes_personal
)
print(
    f"Filas del mes {mes_personal}: esperadas {filas_mes_personal} | obtenidas {filas_con_allow_filtering}"
)
assert filas_con_allow_filtering == filas_mes_personal, (
    f"filas_con_allow_filtering no coincide con las filas del mes {mes_personal}"
)

# Camino paralelo, en el catálogo de esquema: la razón del rechazo es que la clave
# de partición tiene dos columnas, y sin las dos no se puede calcular el token.
clave_de_particion: set[str] = {
    fila.column_name
    for fila in session.execute(
        "SELECT column_name, kind FROM system_schema.columns "
        "WHERE keyspace_name = 'stackoverflow' AND table_name = 'b6_accepted_answers_by_tag'"
    )
    if fila.kind == "partition_key"
}
print(f"Clave de partición: {sorted(clave_de_particion)}")
assert clave_de_particion == {"tag", "year_month"}, (
    "La clave de partición de b6_accepted_answers_by_tag debe ser (tag, year_month)"
)
print("EJERCICIO 3: comprobado")

### EJERCICIO 4: resumen por etiqueta y respondedor

Para que el cuaderno se pueda reejecutar entero, empieza con
`DROP TABLE IF EXISTS b6_tag_answerer_summary`.

Lee `b6_accepted_answers_by_tag` (una fila por respuesta aceptada y etiqueta) y crea `b6_tag_answerer_summary`, con **una fila por etiqueta y respondedor** y clave `((tag), answerer_id)`. Sólo existe la fila de los pares en que el respondedor tuvo alguna respuesta aceptada en esa etiqueta. Sus columnas son:

- `tag` y `answerer_id`: los de la tabla de origen, que ahora identifican la fila;
- `accepted_answers`: cuántas filas (respuestas aceptadas) del respondedor hay en esa etiqueta;
- `total_answer_score`: la suma de `answer_score` de esas respuestas;
- `active_months`: cuántos valores distintos de `year_month` tienen esas respuestas;
- `fastest_minutes`: el menor `minutes_to_answer` de esas respuestas.

Explica en un comentario de la celda por qué la agregación se realiza en el cliente y para cuáles de las cuatro columnas calculadas (`accepted_answers`, `total_answer_score`, `active_months`, `fastest_minutes`) podría mantenerse al día con el tipo `COUNTER` de la sesión 5, y por qué no para las demás.


In [ ]:
# Comprobación del EJERCICIO 4: resumen por etiqueta y respondedor
origen: list[Any] = session.execute(
    "SELECT answer_score FROM stackoverflow.b6_accepted_answers_by_tag"
).all()
assert origen, "b6_accepted_answers_by_tag está vacía: completa antes el ejercicio 2"
assert "b6_tag_answerer_summary" in [
    fila.table_name
    for fila in session.execute(
        "SELECT table_name FROM system_schema.tables WHERE keyspace_name = 'stackoverflow'"
    )
], "Falta b6_tag_answerer_summary"
resumen: list[Any] = session.execute("SELECT * FROM stackoverflow.b6_tag_answerer_summary").all()

# Lo que se conserva al resumir: cada fila de origen cuenta una vez, y su
# puntuación se suma una vez.
aceptadas: int = sum(fila.accepted_answers for fila in resumen)
puntos: int = sum(fila.total_answer_score for fila in resumen)
puntos_origen: int = sum(fila.answer_score or 0 for fila in origen)
print(f"Respuestas aceptadas: origen {len(origen):,} | resumen {aceptadas:,}")
print(f"Puntuación total: origen {puntos_origen:,} | resumen {puntos:,}")
assert aceptadas == len(origen), "La suma de accepted_answers no coincide con las filas de origen"
assert puntos == puntos_origen, "La suma de total_answer_score no coincide con la de origen"

# Camino paralelo, sin agrupar en el cliente: para los pares con más respuestas,
# agregados de CQL sobre cada partición de origen, una por mes del año.
meses: list[str] = [f"{PARAMETROS.año_inicial}-{mes:02d}" for mes in range(1, 13)]
pares: list[Any] = sorted(resumen, key=lambda f: (-f.accepted_answers, f.tag, f.answerer_id))[:20]
for par in pares:
    por_mes: list[Any] = [
        session.execute(
            "SELECT COUNT(*) AS total, SUM(answer_score) AS puntos, "
            "MIN(minutes_to_answer) AS minutos FROM stackoverflow.b6_accepted_answers_by_tag "
            "WHERE tag = %s AND year_month = %s AND answerer_id = %s ALLOW FILTERING",
            (par.tag, mes, par.answerer_id),
        ).one()
        for mes in meses
    ]
    con_respuestas: list[Any] = [fila for fila in por_mes if fila.total > 0]
    clave: tuple[str, int] = (par.tag, par.answerer_id)
    assert par.accepted_answers == sum(fila.total for fila in con_respuestas), (
        f"{clave}: accepted_answers no coincide"
    )
    assert par.total_answer_score == sum(fila.puntos for fila in con_respuestas), (
        f"{clave}: total_answer_score no coincide"
    )
    assert par.active_months == len(con_respuestas), f"{clave}: active_months no coincide"
    assert par.fastest_minutes == min(fila.minutos for fila in con_respuestas), (
        f"{clave}: fastest_minutes no coincide"
    )
print(f"Pares revisados con agregados de CQL: {len(pares)}")
print("EJERCICIO 4: comprobado")

### Las consultas del EJERCICIO 5

La celda siguiente prepara seis consultas sobre `b6_accepted_answers_by_tag` (EJERCICIO 2), cuya clave es `((tag, year_month), answered_at, answer_id)`, y las guarda en el diccionario `CONSULTAS` (número → consulta CQL y sus valores). Los valores salen de tu propia tabla: `ETIQUETA` es la etiqueta con datos en más meses, `MES_1` y `MES_2` sus dos meses más recientes, y `FECHA`, `RESPONDEDOR` y `RESPUESTA` son el `answered_at`, el `answerer_id` y el `answer_id` de una fila de la partición `(ETIQUETA, MES_1)`.


In [ ]:
# Datos del EJERCICIO 5: seis consultas sobre tu tabla del EJERCICIO 2.
from collections import defaultdict

meses_por_etiqueta: defaultdict[str, list[str]] = defaultdict(list)
for fila in session.execute(
    "SELECT DISTINCT tag, year_month FROM stackoverflow.b6_accepted_answers_by_tag"
):
    meses_por_etiqueta[fila.tag].append(fila.year_month)
assert meses_por_etiqueta, "b6_accepted_answers_by_tag está vacía: completa antes el ejercicio 2"
ETIQUETA: str = min(meses_por_etiqueta, key=lambda tag: (-len(meses_por_etiqueta[tag]), tag))
assert len(meses_por_etiqueta[ETIQUETA]) >= 2, "Ninguna etiqueta tiene datos en dos meses"
MES_1, MES_2 = sorted(meses_por_etiqueta[ETIQUETA], reverse=True)[:2]

# Una fila de en medio de la partición (ETIQUETA, MES_1), que sale ordenada por fecha.
particion: list[Any] = session.execute(
    "SELECT answered_at, answer_id, answerer_id FROM stackoverflow.b6_accepted_answers_by_tag "
    "WHERE tag = %s AND year_month = %s",
    (ETIQUETA, MES_1),
).all()
referencia: Any = particion[len(particion) // 2]
FECHA, RESPONDEDOR, RESPUESTA = referencia.answered_at, referencia.answerer_id, referencia.answer_id

SELECT_BASE: str = "SELECT answer_id FROM stackoverflow.b6_accepted_answers_by_tag WHERE "
CONSULTAS: dict[int, tuple[str, tuple]] = {
    1: (SELECT_BASE + "tag = %s AND year_month = %s", (ETIQUETA, MES_1)),
    2: (SELECT_BASE + "tag = %s", (ETIQUETA,)),
    3: (
        SELECT_BASE + "tag = %s AND year_month = %s AND answered_at >= %s",
        (ETIQUETA, MES_1, FECHA),
    ),
    4: (
        SELECT_BASE + "tag = %s AND year_month = %s AND answerer_id = %s",
        (ETIQUETA, MES_1, RESPONDEDOR),
    ),
    5: (
        SELECT_BASE + "tag = %s AND year_month = %s AND answer_id = %s",
        (ETIQUETA, MES_1, RESPUESTA),
    ),
    6: (SELECT_BASE + "tag = %s AND year_month IN (%s, %s)", (ETIQUETA, MES_1, MES_2)),
}
print(f"ETIQUETA={ETIQUETA!r} MES_1={MES_1!r} MES_2={MES_2!r}")
print(f"FECHA={FECHA} RESPONDEDOR={RESPONDEDOR} RESPUESTA={RESPUESTA}")
for numero, (cql, valores) in CONSULTAS.items():
    print(f"{numero}: ... WHERE {cql.removeprefix(SELECT_BASE)}")

### EJERCICIO 5: qué consultas sirve la clave

Estas son las seis consultas de `CONSULTAS`, preparadas por la celda anterior:

| Nº | Filtro |
|---|---|
| 1 | `tag = ETIQUETA AND year_month = MES_1` |
| 2 | `tag = ETIQUETA` |
| 3 | `tag = ETIQUETA AND year_month = MES_1 AND answered_at >= FECHA` |
| 4 | `tag = ETIQUETA AND year_month = MES_1 AND answerer_id = RESPONDEDOR` |
| 5 | `tag = ETIQUETA AND year_month = MES_1 AND answer_id = RESPUESTA` |
| 6 | `tag = ETIQUETA AND year_month IN (MES_1, MES_2)` |

**Antes de ejecutarlas**, decide para cada una si Cassandra la acepta sin `ALLOW FILTERING` y guarda tu predicción en el diccionario `prediccion` (`dict[int,bool]`), con las claves 1 a 6 y el valor `True` si la acepta o `False` si la rechaza. Ejecuta después cada consulta de `CONSULTAS` tal cual, dentro de un `try`/`except Exception as exc`, y guarda en el diccionario `resultado`, con las claves 1 a 6, el número de filas que devuelve o, si la rechaza, `type(exc).__name__`. La comprobación contrasta tu predicción con un *hash* de las respuestas correctas, así que no las revela: sólo dice si has acertado las seis. Si no, no cambies valores a ciegas: busca en qué te equivocaste y explícalo, porque es lo que tendrás que defender.

Explica en un comentario de la celda, para cada consulta rechazada, por qué la rechaza y qué clave tendría una tabla que la sirviera sin `ALLOW FILTERING`. En una de las rechazadas, añadir `ALLOW FILTERING` sería aceptable: ¿en cuál, y por qué?


In [ ]:
# Comprobación del EJERCICIO 5: qué consultas sirve la clave
import hashlib

assert "prediccion" in globals(), "Guarda tu predicción en el diccionario prediccion"
assert set(prediccion) == set(CONSULTAS), "prediccion debe tener las claves 1 a 6"
assert all(isinstance(valor, bool) for valor in prediccion.values()), (
    "Cada predicción debe ser True (la acepta) o False (la rechaza)"
)
assert "resultado" in globals(), "Guarda lo que devuelve cada consulta en el diccionario resultado"
assert set(resultado) == set(CONSULTAS), "resultado debe tener las claves 1 a 6"
assert all(isinstance(valor, int) or valor == "InvalidRequest" for valor in resultado.values()), (
    "Cada valor de resultado es un número de filas o type(exc).__name__ (InvalidRequest)"
)

# Camino paralelo para las filas: las particiones completas de ETIQUETA, leídas por su
# clave, y lo que devolvería cada filtro, contado en el cliente.
filas_etiqueta: list[Any] = [
    fila
    for mes in meses_por_etiqueta[ETIQUETA]
    for fila in session.execute(
        "SELECT year_month, answered_at, answer_id, answerer_id "
        "FROM stackoverflow.b6_accepted_answers_by_tag WHERE tag = %s AND year_month = %s",
        (ETIQUETA, mes),
    )
]
mes_1: list[Any] = [fila for fila in filas_etiqueta if fila.year_month == MES_1]
filas_esperadas: dict[int, int] = {
    1: len(mes_1),
    2: len(filas_etiqueta),
    3: sum(1 for fila in mes_1 if fila.answered_at >= FECHA),
    4: sum(1 for fila in mes_1 if fila.answerer_id == RESPONDEDOR),
    5: sum(1 for fila in mes_1 if fila.answer_id == RESPUESTA),
    6: sum(1 for fila in filas_etiqueta if fila.year_month in (MES_1, MES_2)),
}

# Lo guardado se contrasta con Cassandra sin anunciar qué consultas acepta: sólo se
# avisa cuando resultado no es lo que da la consulta ejecutada tal cual.
for numero, (cql, valores) in CONSULTAS.items():
    obtenido: int | str
    try:
        obtenido = len(session.execute(cql, valores).all())
    except Exception as error_consulta:
        obtenido = type(error_consulta).__name__
    assert resultado[numero] == obtenido, (
        f"Consulta {numero}: resultado no es lo que devuelve al ejecutarla tal cual, "
        "sin ALLOW FILTERING"
    )
    if isinstance(obtenido, int):
        assert obtenido == filas_esperadas[numero], (
            f"Consulta {numero}: no devuelve las filas de la partición que le corresponden"
        )
print(f"Consultas contrastadas con Cassandra: {len(CONSULTAS)}")

# La predicción se compara con el SHA-256 de las respuestas correctas, no con las
# respuestas: la celda no las contiene, y el aviso no dice qué consulta falla.
SHA256_PREDICCION: str = "e68369e0c3f39a70fe394173104bf627e739da7900ed967dbc36ff1b9b10231a"
clave_prediccion: str = "bdge-cassandra-sesion6-ejercicio5:" + "".join(
    "1" if prediccion[numero] else "0" for numero in sorted(CONSULTAS)
)
assert hashlib.sha256(clave_prediccion.encode("utf-8")).hexdigest() == SHA256_PREDICCION, (
    "Alguna de tus predicciones no es lo que hace Cassandra: repasa la clave de la tabla"
)
print("EJERCICIO 5: comprobado")

### EJERCICIO 6: temas más activos con su mejor respondedor

Para que el cuaderno se pueda reejecutar entero, empieza con
`DROP TABLE IF EXISTS b6_hot_tags`.

Desde `b6_tag_answerer_summary` (una fila por etiqueta y respondedor), agrupa en el cliente las filas de cada etiqueta y crea `b6_hot_tags`, con **una fila por etiqueta**. Sus columnas son:

- `fingerprint`: tu `HUELLA`, que es la clave de partición y da a cada alumno su propia partición;
- `accepted_total`: la suma de `accepted_answers` de todos los respondedores de la etiqueta;
- `tag`: la etiqueta;
- `distinct_answerers`: cuántos respondedores (filas de origen) tiene la etiqueta;
- `best_answerer_id`: el respondedor con más `accepted_answers` en la etiqueta; si empatan varios, el de menor `answerer_id`;
- `best_accepted`, `best_total_score` y `best_fastest_minutes`: los valores de `accepted_answers`, `total_answer_score` y `fastest_minutes` de la fila de origen de ese respondedor.

**Las columnas de *clustering* las diseñas tú.** La tabla existe para una sola consulta, la que hará el EJERCICIO 7: leer tu partición con `WHERE fingerprint = ?`, sin `ORDER BY` y sin ordenar en el cliente, y que las filas salgan ya en el orden del ranking, con el tema de más respuestas aceptadas (`accepted_total`) primero y, entre los que empaten, por etiqueta en orden alfabético. Elige las columnas de *clustering* y su orden (`WITH CLUSTERING ORDER BY`) para que así sea. Recuerda que la clave primaria identifica la fila.

Explica en un comentario de la celda qué clave has elegido, por qué sirve a esa consulta y qué pasaría con las etiquetas que empatan en `accepted_total` si la clave no las distinguiera.


In [ ]:
# Comprobación del EJERCICIO 6: temas más activos con su mejor respondedor
resumen: list[Any] = session.execute(
    "SELECT accepted_answers FROM stackoverflow.b6_tag_answerer_summary"
).all()
assert resumen, "b6_tag_answerer_summary está vacía: completa antes el ejercicio 4"

# La partición es la del enunciado; el clustering es el que hayas diseñado.
particion_hot_tags: list[str] = [
    fila.column_name
    for fila in session.execute(
        "SELECT column_name, kind FROM system_schema.columns "
        "WHERE keyspace_name = 'stackoverflow' AND table_name = 'b6_hot_tags'"
    )
    if fila.kind == "partition_key"
]
assert particion_hot_tags == ["fingerprint"], (
    "La clave de partición de b6_hot_tags debe ser sólo fingerprint"
)
temas: list[Any] = session.execute(
    "SELECT * FROM stackoverflow.b6_hot_tags WHERE fingerprint = %s", (HUELLA,)
).all()

# Lo que se conserva: una fila por cada etiqueta de origen, cada respondedor
# cuenta una vez en su etiqueta y cada respuesta aceptada, una vez.
etiquetas_origen: set[str] = {
    fila.tag
    for fila in session.execute("SELECT DISTINCT tag FROM stackoverflow.b6_tag_answerer_summary")
}
print(f"Etiquetas: origen {len(etiquetas_origen):,} | obtenidas {len(temas):,}")
assert {fila.tag for fila in temas} == etiquetas_origen, (
    "Las etiquetas no coinciden: si faltan, revisa si tu clave primaria distingue a dos "
    "etiquetas distintas"
)
assert len(temas) == len(etiquetas_origen), "Hay etiquetas repetidas en tu partición"
assert sum(fila.distinct_answerers for fila in temas) == len(resumen), (
    "La suma de distinct_answerers no coincide con las filas de b6_tag_answerer_summary"
)
assert sum(fila.accepted_total for fila in temas) == sum(f.accepted_answers for f in resumen), (
    "La suma de accepted_total no coincide con la de accepted_answers"
)

# Camino paralelo, sin agrupar en el cliente: cada fila se revalida contra la
# partición de su etiqueta, con agregados de CQL y consultas puntuales.
for tema in temas:
    agregados: Any = session.execute(
        "SELECT SUM(accepted_answers) AS total, COUNT(*) AS respondedores, "
        "MAX(accepted_answers) AS maximo FROM stackoverflow.b6_tag_answerer_summary "
        "WHERE tag = %s",
        (tema.tag,),
    ).one()
    assert tema.accepted_total == agregados.total, f"Etiqueta {tema.tag}: accepted_total"
    assert tema.distinct_answerers == agregados.respondedores, (
        f"Etiqueta {tema.tag}: distinct_answerers"
    )
    assert tema.best_accepted == agregados.maximo, (
        f"Etiqueta {tema.tag}: best_accepted no es el máximo de la etiqueta"
    )

    mejor: Any = session.execute(
        "SELECT accepted_answers, total_answer_score, fastest_minutes "
        "FROM stackoverflow.b6_tag_answerer_summary WHERE tag = %s AND answerer_id = %s",
        (tema.tag, tema.best_answerer_id),
    ).one()
    assert mejor is not None, f"Etiqueta {tema.tag}: best_answerer_id no es de esta etiqueta"
    assert (tema.best_accepted, tema.best_total_score, tema.best_fastest_minutes) == (
        mejor.accepted_answers,
        mejor.total_answer_score,
        mejor.fastest_minutes,
    ), f"Etiqueta {tema.tag}: las columnas best_* no son las de best_answerer_id"

    # Ante un empate gana el menor id: nadie con un id menor llega a ese máximo.
    empatados_antes: int = (
        session.execute(
            "SELECT COUNT(*) AS total FROM stackoverflow.b6_tag_answerer_summary "
            "WHERE tag = %s AND answerer_id < %s AND accepted_answers = %s ALLOW FILTERING",
            (tema.tag, tema.best_answerer_id, tema.best_accepted),
        )
        .one()
        .total
    )
    assert empatados_antes == 0, (
        f"Etiqueta {tema.tag}: el empate se resuelve a favor del menor answerer_id"
    )
print(f"Etiquetas revisadas contra su partición: {len(temas):,}")

# La partición debe salir ya en el orden del ranking, sin ORDER BY: más respuestas
# aceptadas primero y, en los empates, por etiqueta.
claves_leidas: list[tuple[int, str]] = [(f.accepted_total, f.tag) for f in temas]
assert claves_leidas == sorted(claves_leidas, key=lambda c: (-c[0], c[1])), (
    "Tu partición no sale en el orden del ranking: revisa las columnas de clustering y su orden"
)
print("EJERCICIO 6: comprobado")

### EJERCICIO 7: resultado final Cassandra

Para que el cuaderno se pueda reejecutar entero, empieza con
`DROP TABLE IF EXISTS b6_submission`.

Lee exclusivamente la partición personal de `b6_hot_tags` (una fila por etiqueta, ya ordenada por el *clustering*) y escribe las `PARAMETROS.limite` primeras en `b6_submission`, con **una fila por posición del ranking** y clave `((fingerprint), position)`. Si hubiera menos etiquetas, se escriben las que haya. Sus columnas son:

- `fingerprint`: tu `HUELLA`;
- `position`: el puesto en el ranking, de 1 en adelante;
- `tag`, `accepted_total`, `distinct_answerers`, `best_answerer_id`, `best_accepted`, `best_total_score` y `best_fastest_minutes`: copiadas de la fila de esa etiqueta en `b6_hot_tags`.

Orden: `accepted_total DESC`, `tag ASC`.

Mira después el resultado y contesta en un comentario: ¿el mejor respondedor de un tema lo es también de los demás? ¿Por qué basta leer una sola partición con `LIMIT` para obtener el ranking, y qué ocurriría si la clave de `b6_hot_tags` fuera `((fingerprint), tag, accepted_total)`?


In [ ]:
# Comprobación del EJERCICIO 7: resultado final Cassandra
temas: list[Any] = session.execute(
    "SELECT * FROM stackoverflow.b6_hot_tags WHERE fingerprint = %s", (HUELLA,)
).all()
assert temas, "b6_hot_tags está vacía: completa antes el ejercicio 6"

top: list[Any] = sorted(temas, key=lambda f: (-f.accepted_total, f.tag))[: PARAMETROS.limite]

entrega: list[Any] = sorted(
    session.execute(
        "SELECT * FROM stackoverflow.b6_submission WHERE fingerprint = %s", (HUELLA,)
    ).all(),
    key=lambda f: f.position,
)
print(f"tag esperado: {[f.tag for f in top]}")
print(f"tag obtenido: {[f.tag for f in entrega]}")
assert len(entrega) == len(top), f"Deberían ser {len(top)} filas"
assert [f.position for f in entrega] == list(range(1, len(entrega) + 1)), (
    "position debe ser un orden total 1..N sin huecos"
)
assert [f.tag for f in entrega] == [f.tag for f in top], (
    "No son las mismas etiquetas, o no en el orden del enunciado"
)
for esperada, obtenida in zip(top, entrega):
    assert (
        esperada.accepted_total,
        esperada.distinct_answerers,
        esperada.best_answerer_id,
        esperada.best_accepted,
        esperada.best_total_score,
        esperada.best_fastest_minutes,
    ) == (
        obtenida.accepted_total,
        obtenida.distinct_answerers,
        obtenida.best_answerer_id,
        obtenida.best_accepted,
        obtenida.best_total_score,
        obtenida.best_fastest_minutes,
    ), f"Etiqueta {esperada.tag}: alguna columna no coincide con b6_hot_tags"
print("EJERCICIO 7: comprobado")

## Resultado que se registrará

Se relee exclusivamente la partición personal de `b6_submission`.


In [ ]:
import json
from decimal import Decimal

CONSULTA_FINAL: str = (
    "SELECT position, tag, accepted_total, distinct_answerers, best_answerer_id, "
    "best_accepted, best_total_score, best_fastest_minutes "
    "FROM stackoverflow.b6_submission WHERE fingerprint = ? ORDER BY position"
)

tablas_b6: set[str] = {
    row.table_name
    for row in session.execute(
        "SELECT table_name FROM system_schema.tables WHERE keyspace_name = %s", ("stackoverflow",)
    )
}
if "b6_submission" not in tablas_b6:
    print("Falta b6_submission: completa los ejercicios anteriores antes de registrar la entrega.")
    RESULTADO_FINAL: list[dict[str, object]] = []
else:
    rows: ResultSet = session.execute(session.prepare(CONSULTA_FINAL), [HUELLA])
    RESULTADO_FINAL = [row._asdict() for row in rows]

    def _valor_json(value: object) -> int | float | str:
        if isinstance(value, Decimal):
            return int(value) if value == value.to_integral_value() else float(value)
        if hasattr(value, "isoformat"):
            return value.isoformat()
        raise TypeError(f"Valor no serializable: {type(value).__name__}")

    RESULTADO_FINAL = json.loads(
        json.dumps(RESULTADO_FINAL, ensure_ascii=False, allow_nan=False, default=_valor_json)
    )
    for row in RESULTADO_FINAL:
        print(row)

## Entrega del resultado

El envío registra exactamente `RESULTADO_FINAL`, que se ha releído de la
estructura final de esta base de datos. Ejecuta esta celda una sola vez cuando
hayas terminado el boletín. Si la vuelves a ejecutar en el mismo kernel, no
creará una segunda entrega. Con el identificador de ejemplo no se contacta con
el servidor.

El cliente ya se descargó en «Preparación de la entrega», así que no hay
que copiar ningún ZIP junto al cuaderno.

In [ ]:
recibo_entrega: evidence.EvidenceReceipt | None
if not RESULTADO_FINAL:
    recibo_entrega = None
    print("No hay resultado final que enviar: completa los ejercicios anteriores.")
elif evidence.is_example_identifier(DNI):
    recibo_entrega = None
    print("DNI de ejemplo: no se ha enviado ninguna entrega. Cambia DNI antes de entregar.")
elif globals().get("recibo_entrega") is not None:
    print("La entrega ya se registró en este kernel; no se ha creado otra.")
else:
    recibo_entrega = evidence.submit_final_result_unless_example(
        course=CURSO,
        notebook=BOLETIN,
        student_identifier=DNI,
        query_text=CONSULTA_FINAL,
        result=RESULTADO_FINAL,
    )
recibo_entrega

## Comprobación de la entrega

Esta celda sólo consulta el último registro de este curso, boletín e
identificador. No vuelve a enviar `RESULTADO_FINAL` ni modifica SQLite, así que
puedes ejecutarla cuantas veces quieras: mientras no hagas un nuevo envío,
devolverá el mismo ID y la misma fecha UTC. El ID es el de la fila que el
profesor puede localizar en la base SQLite.

In [ ]:
estado_entrega: evidence.SubmissionStatus | None
if evidence.is_example_identifier(DNI):
    estado_entrega = None
    print("DNI de ejemplo: se omite la consulta de una entrega real.")
else:
    estado_entrega = evidence.check_submission_status(
        course=CURSO,
        notebook=BOLETIN,
        student_identifier=DNI,
    )
estado_entrega

In [ ]:
# @title Diapositiva: Para el examen: qué demuestra el resultado final
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #8a6a00;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#8a6a00" size="8" noshade style="display:none"><div style="color:#8a6a00;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#8a6a00" style="color:inherit"><b>Para el examen</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Para el examen: qué demuestra el resultado final</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>El resultado final depende de transformaciones materializadas anteriores</li><li style=\'margin:10px 0;list-style-position:outside\'>La granularidad elegida determina qué agregaciones son correctas</li><li style=\'margin:10px 0;list-style-position:outside\'>Los nulos y los desempates forman parte del contrato del resultado</li><li style=\'margin:10px 0;list-style-position:outside\'>El DNI parametriza los datos sin salir del cuaderno</li><li style=\'margin:10px 0;list-style-position:outside\'>Cada diseño se justifica con la consulta que sirve, no con intuiciones</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #8a6a00;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Pregunta típica: explicar qué tabla lee el resultado final y por qué basta una partición</div></blockquote><hr color="#8a6a00" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Lo que deja la sesión 6
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #287681;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#287681" size="8" noshade style="display:none"><div style="color:#287681;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#287681" style="color:inherit"><b>Recapitulación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Lo que deja la sesión 6</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Las tablas derivadas sustituyen a los <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>JOIN</code>, y alguien tiene que mantenerlas</li><li style=\'margin:10px 0;list-style-position:outside\'>Otra clave de acceso (tabla propia, índice o vista) se paga siempre en escritura</li><li style=\'margin:10px 0;list-style-position:outside\'><code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>ALLOW FILTERING</code> no es una solución de diseño</li><li style=\'margin:10px 0;list-style-position:outside\'>TTL y escrituras condicionales cambian la semántica de la escritura</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #287681;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>Tres modelos vistos: tablas normalizadas, documentos y tablas por consulta</div></blockquote><hr color="#287681" size="2" noshade style="display:none"></div>'))

In [ ]:
# @title Diapositiva: Sesión 7: Neo4j
from IPython.display import HTML, display; display(HTML('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Atkinson+Hyperlegible:ital,wght@0,400;0,700;1,400;1,700&amp;display=swap"><div style="box-sizing:border-box;width:100%;max-width:960px;min-height:420px;margin:20px auto;padding:34px 46px;background:#fff9db;color:#29261d;border:1px solid #dfcc78;border-top:10px solid #c69200;border-radius:10px;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:22px;line-height:1.42;text-align:left;overflow:hidden"><hr color="#c69200" size="8" noshade style="display:none"><div style="color:#c69200;font-size:15px;font-weight:bold;letter-spacing:1px;margin-bottom:18px;text-transform:uppercase"><font color="#c69200" style="color:inherit"><b>A continuación</b></font></div><h1 style="color:#7b2c19;font-family:\'Atkinson Hyperlegible\',Inter,Aptos,\'Segoe UI\',Helvetica,Arial,sans-serif;font-size:34px;line-height:1.15;margin:0 0 22px">Sesión 7: Neo4j</h1><ul style=\'margin:18px 0 8px;padding-left:30px\'><li style=\'margin:10px 0;list-style-position:outside\'>Los mismos datos como grafo: nodos, relaciones y <em>constraints</em></li><li style=\'margin:10px 0;list-style-position:outside\'>Carga por lotes con <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>UNWIND</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>MERGE</code>, sin duplicar</li><li style=\'margin:10px 0;list-style-position:outside\'>Cypher: patrones, agregaciones y recorridos de varios saltos</li><li style=\'margin:10px 0;list-style-position:outside\'>Planes con <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>EXPLAIN</code> y <code style=\'background:#fff0ae;color:#713019;padding:2px 6px;border-radius:4px;font-size:0.9em\'>PROFILE</code></li><li style=\'margin:10px 0;list-style-position:outside\'>El boletín evaluable de Neo4j</li></ul><blockquote style=\'background:#fff0ae;color:inherit;border:none;border-left:6px solid #c69200;margin:22px 0 0;padding:12px 16px\'><div style=\'margin-top:0px\'>El valor del grafo está en los recorridos que hace baratos</div></blockquote><hr color="#c69200" size="2" noshade style="display:none"></div>'))

## Guía de estudio para el examen

Con lo de esta sesión y lo de la sesión 5, debes ser capaz de hacer esto **sin IA y sin el cuaderno delante**:

- Escribir la `PRIMARY KEY` que sirve a una consulta dada, separando partición y *clustering*
- Decir qué consultas puede responder una tabla a partir de su clave, y cuáles no
- Explicar qué hace `ALLOW FILTERING` y por qué no es una solución
- Reconocer cuándo un índice es mala idea y qué tabla lo sustituye
- Decidir entre un índice SAI y una tabla dirigida por la consulta, y justificarlo
- Decidir entre vista materializada y tabla mantenida por la aplicación, y justificarlo (está en la teoría)
- Convertir una consulta SQL con JOIN en el conjunto de tablas que Cassandra necesita
- Explicar por qué toda escritura es un *upsert* y qué cambia con `IF NOT EXISTS`
- Decir qué garantiza y qué cuesta una escritura condicional (Paxos)
- Usar TTL, `TTL()` y `WRITETIME()`, y explicar por qué el TTL por columna deja filas parciales
- Elegir niveles de consistencia de lectura y escritura y comprobar `R + W > RF`
- Explicar por qué `BATCH` no es una optimización de rendimiento y en qué caso sí tiene sentido usarlo
- Explicar qué es una *tombstone* y por qué acumularlas degrada las lecturas
- Explicar por qué `GROUP BY` sólo funciona dentro de una partición y dónde se agrega entonces

Si al intentar cualquiera de estos puntos te quedas en blanco, vuelve al ejercicio o al ejemplo correspondiente de este cuaderno antes del examen.

## Referencias

- [Documentación oficial de Apache Cassandra 5.0](https://cassandra.apache.org/doc/5.0/index.html)
- [Referencia de CQL](https://cassandra.apache.org/doc/5.0/cassandra/developing/cql/index.html)
- [Modelado de datos en Cassandra](https://cassandra.apache.org/doc/5.0/cassandra/developing/data-modeling/index.html)
- [Compresión de tablas](https://cassandra.apache.org/doc/5.0/cassandra/managing/operating/compression.html)
- [Índices SAI](https://cassandra.apache.org/doc/5.0/cassandra/developing/cql/indexing/sai/sai-overview.html)
- [Documentación del *driver* de Python](https://docs.datastax.com/en/developer/python-driver/latest/), en particular los [perfiles de ejecución](https://docs.datastax.com/en/developer/python-driver/latest/execution_profiles/) y la [paginación](https://docs.datastax.com/en/developer/python-driver/latest/query_paging/)
- [Repositorio del *driver* de Python](https://github.com/apache/cassandra-python-driver), hoy un proyecto de la Apache Software Foundation